# 03_embedding — 임베딩·δ·d_flow·중력 대조군 코드 검산과 입력 준비 (S4 1부, 0S-4)

S4 1부다. **실데이터 임베딩, 실데이터 δ·δ̄, 판정은 계산하지 않는다**(`RUN_REAL = False`). 이 노트북과 산출물을 고정 커밋한 뒤, 2부에서 이 플래그만 True로 바꿔 실행한다.

- 1절: 판정 계산 코드(`embed_lib.py`)를 합성 자료로 검산한다. tree(깊이 4)를 S1b 값으로 다시 맞추고, δ fixture(`tree`, `c4`, `square4`, 거리에 g_i + g_j를 더해도 불변), `gravity_grid`(d_flow), 추정 OD 거부, 중력 적합, 재실행 규칙, 판정 함수 대표 사례를 확인한다. 하나라도 실패하면 이 노트북은 거기서 멈춘다.
- 2절: 입력 준비(거리 행렬만). `d_time` 250·249·247개 단위, 관측 OD(249개 단위) 요약과 `d_flow`(주·대안, 폐포), 중력 대조군 20개, 섬 연결 지역. 행렬은 `C:\dev\kfm-data\spike\s4\`, 고정 값은 `s4_freeze.json`에 남긴다.
- 3절: 실데이터 δ·δ̄·임베딩·판정. 코드만 있고 건너뛴다. 셀은 `embed_lib` 함수만 부르며, 같은 함수의 배선을 합성 장난감 자료로 1절에서 점검했다(실데이터 값 아님).

실행 환경은 `C:\dev\kfm-spike`, 실행은 `jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=36000`이다. 0S 탐색 코드라서 `kfm` 패키지·`runs`·스키마 규칙 밖이다(PLAN 6.1a).

## 판정 규칙 요약 (ADR 0010 2·5·6·7항, ADR 0013 4항. 이 노트북 실행 결과로 바꾸지 않는다)
- 임베딩: 상대 stress Σ((s·p − d)/d)², 척도 s는 매 단계 닫힌 해. 2차원 푸앵카레 원판(곡률 −1, R_max = 5)과 2차원 유클리드. 2단계 최적화(RiemannianAdam → L-BFGS-B). 재시작 쌍곡 5(유클리드 해를 줄인 2 + U(±0.1) 3), 유클리드 5(고전 MDS 1 + U(±0.1) 4).
- "쌍곡 이점": 마무리가 수렴한 재시작 중 상대 stress 최소의 평균 상대 왜곡 비(쌍곡 / 유클리드) ≤ 0.8. `d_flow`는 대조군 5개 비의 최솟값보다도 작아야 한다. 대조군 중 하나라도 판정 재시작이 없으면 "보류". 쌍곡 최솟값이 유클리드 최솟값 × (1 + 10⁻³) 이내이면 "쌍곡 이점 없음(평탄 극한)", 그보다 크면 최적화 실패(쌍곡 U(±0.1) 재시작 3개만 시드 +5, +10으로 최대 2회 다시 돌리고 쌍곡은 모든 시도의 재시작을 합쳐 판정, 두 번 재실행해도 실패이면 "최적화 실패"와 "보류").
- "중력모형 이상의 위계": δ̄(`d_flow`)가 대조군 20개보다 모두 작아야 한다. 주·대안 정규화 각각, 전체와 섬 연결 지역 제외 부분집합 모두에서(4개 조건).
- `d_flow`: ℓ_ij = −log(T_ij / √(O_i·D_j)), ℓ_sym = 두 방향 평균, 한 방향이라도 0이면 엣지 없음, 최단경로 폐포. 대안 ℓ′ = −log((T_ij + T_ji)/√(V_i·V_j))는 같은 엣지 집합. 관측 모드는 추정 OD를 거부한다. 연결 성분이 둘 이상이면 오류.
- 대조군: 이중 제약 지수 모형(O_i, D_j, 가중 평균 방향 통행시간에 맞춘 β, Furness) → 존 간 합 × 0.2(반올림)를 총량으로 다항 재표본 → ×5 → 관측 0 패턴. 시드 0, 1, 2, …에서 연결된 20개를 모으고 건너뛴 시드를 기록한다.

## 고정 전 결정 사항과 구현 해석 (2026-10-19 결정 반영: ADR 0010 2항·ADR 0013 4항 보강)
1. **재시작 재실행 (ADR 0013 4항 보강)**: 첫 시도 = 유클리드 5(고전 MDS 1 + U 시드 0..3) + 쌍곡 5(줄인 초기값 2 + U 시드 0..2). 쌍곡 최소 stress가 유클리드 최소 stress × (1 + 10⁻³)보다 크면 **쌍곡의 U(±0.1) 재시작 3개만** U 시드에 +5를, 그래도 실패이면 +10을 더해 다시 돌린다(최대 2회, 쌍곡 U 시드 5..7, 10..12). 유클리드 결과와 줄인 초기값 2개는 다시 돌리지 않고 그대로 둔다. **쌍곡은 모든 시도의 재시작을 합쳐 판정**하고(첫 시도 5개 + 재실행분 3개씩), 실패 판정은 시도마다 그때까지 합친 쌍곡 최솟값으로 한다. 두 번 재실행해도 실패이면 기록은 "최적화 실패", 쌍곡 이점은 "보류"다.
2. **실패와 평탄 극한의 최솟값**: 마무리 수렴 여부와 관계없는 최소 stress로 가른다(ADR 0013: 평탄 극한 기록은 수렴을 요구하지 않는다). 쌍곡 이점의 비는 합친 재시작 중 마무리가 수렴한 것의 최소 stress 재시작으로만 구한다.
3. **d_flow 단위: 제주시·서귀포시 제외(247개 단위) (ADR 0010 2항 보강)**: "통근·통학 OD처럼 바다를 건너는 일상 통행이 거의 없는 자료에서는 섬 연결 지역 중 본토와 양방향 통행이 없는 지역도 도서지역 제외로 미리 뺀다(제주시·서귀포시)." 자료 사실은 이렇다. 2020 통근·통학 OD(249개 단위)에서 제주시·서귀포시는 서로 사이에만 쌍방향 통행이 있고 육지와는 한 방향 통행(인천 남동구 → 제주시 9명)뿐이라, "한 방향이라도 0이면 엣지 없음" 규칙 아래 `d_flow` 그래프의 연결 성분이 둘이 된다. 구현(`flow_exclusion`)은 제외를 쌍방향 그래프의 본토 성분 밖이면서 섬 연결 지역(ADR 5항)인 지역으로 한정하고, 그 밖의 이유로 끊기면 중단하며, 뺀 지역은 목록으로 기록한다. 이 경우 `d_flow`의 "전체"는 247개 단위, "섬 제외"는 울릉군까지 뺀 246개 단위다. 같은 단위끼리 비교하려고 `d_time`도 247개 단위 부분행렬로 함께 낸다.
4. **대조군 적합 단위**: β, O_i, D_j, 재표본 총량(존 간 합 × 0.2)은 제외 후 247개 단위의 관측 OD와 방향 통행시간(하위 지역 쌍 인구 가중 평균, 249개 단위에서 부분행렬)에서 구한다.
5. **임베딩 대상**: `d_time` 250개 단위, `d_time` 247개 단위(`d_flow`와 같은 단위), `d_flow` 주·대안, 대조군 앞 5개의 주·대안(14개 입력)으로 한다. `d_time` 249개 단위는 δ만 낸다. 그래프 재구성 지표(MAP, DESIGN 6.6)는 0S-4 범위 밖이라 넣지 않는다.

In [1]:
# 03_embedding 설정 셀 (S4 1부): 경로, 설정값, 환경, 보조 함수
import os

for _name in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_name] = "1"  # numpy, torch를 불러오기 전에 고정한다(자식 프로세스가 물려받는다)
os.environ["PYTORCH_JIT"] = "0"  # torch JIT 프로파일링 실행기는 호출 순서에 따라 부동소수점 마지막 자리를 바꾼다. 끄면 재시작 결과가 시드와 입력으로만 정해진다.

import csv
import datetime
import hashlib
import io
import itertools
import json
import math
import platform
import subprocess
import sys
import tempfile
import time
import warnings
from pathlib import Path

import geoopt
import numpy as np
import pandas as pd
import scipy
import torch
from IPython.display import display

import embed_lib as el

assert not torch.jit._state._enabled.enabled, "torch JIT가 켜져 있다(PYTORCH_JIT=0이 torch import보다 먼저여야 한다)"
RUN_REAL = True    # 3절(실데이터 δ·임베딩·판정)을 돌릴지. 2부에서 이 값만 True로 바꿔 실행한다.
N_WORKERS = 10      # 재시작 병렬 프로세스 수(각 1스레드). 결과는 시드로 결정되므로 이 값에 의존하지 않는다.
M1, M2 = 0.5, 0.9   # 줄인 유클리드 해의 최대 노름 (ADR 0013 4항: 0.1*sqrt(2) <= m1 < m2 < tanh 2.5 = 0.9866). 실데이터 실행 전에 커밋한다.

CFG = {**el.CFG, "m1": M1, "m2": M2}
el.check_cfg(CFG)

T_START = time.time()
RAW = Path(r"C:\dev\kfm-data\raw")
DATA = Path(r"C:\dev\kfm-data\spike")
OUT = DATA / "s4"
OUT.mkdir(exist_ok=True)
NB_DIR = Path.cwd()  # notebooks/spike (nbconvert는 노트북 폴더에서 실행한다)
assert (NB_DIR / "embed_lib.py").exists(), "notebooks/spike 폴더에서 실행해야 한다"
assert "kfm-spike" in sys.executable, "커널이 kfm-spike 환경이 아니다"
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 120)

print("실행 시각:", datetime.datetime.now().isoformat(timespec="seconds"), "| python", sys.version.split()[0], "|", platform.platform())
print("실행 파일:", sys.executable, "| RUN_REAL =", RUN_REAL, "| N_WORKERS =", N_WORKERS)
print("torch", torch.__version__, "| geoopt", geoopt.__version__, "| numpy", np.__version__, "| scipy", scipy.__version__,
      "| pandas", pd.__version__, "| numba", el.numba.__version__, "| 스레드(torch)", torch.get_num_threads())
print("설정:", CFG)
print("config_hash:", el.config_hash(CFG))

SEED_RULES = {
    "euclid_restarts": "고전 MDS(최대 노름 0.1*sqrt(2)로 줄임) 1 + U(±0.1) 시드 0..3",
    "poincare_restarts": "유클리드 최소 stress 해를 무게중심 원점으로 옮긴 뒤 최대 노름 m1, m2로 줄인 2 + U(±0.1) 시드 0..2",
    "same_seed_same_init": "U 초기값 = np.random.default_rng(seed).uniform(-0.1, 0.1, (n, 2)). 같은 시드는 두 공간에서 같다",
    "attempt": "첫 시도 = 유클리드 5 + 쌍곡 5. 재실행분 = 쌍곡 U(±0.1) 재시작 3개(유클리드와 줄인 초기값 2개는 다시 돌리지 않음)",
    "retry": "그때까지 합친 모든 쌍곡 재시작의 최소 stress > 유클리드 최소 stress x (1 + 1e-3)이면 쌍곡의 U(±0.1) 재시작 3개만 U 시드에 +5, 그래도 실패면 +10을 더해 다시 돌린다(최대 2회). "
             "쌍곡은 모든 시도의 재시작을 합쳐 판정하고 실패 판정은 시도마다 합친 쌍곡 최솟값으로 한다. 두 번 재실행해도 실패면 기록 '최적화 실패'·쌍곡 이점 '보류'",
    "judgement_restart": "마무리(L-BFGS-B)가 수렴한 재시작 중 상대 stress 최소. 없으면 보류. 평탄 극한·실패 판정의 최솟값은 수렴 여부와 무관",
    "control_resample_rng": f"np.random.default_rng([{el.RESAMPLE_STREAM}, r]) (임베딩 재시작의 default_rng(seed)와 다른 난수열)",
    "control_seeds": "r = 0, 1, 2, ... 연결 성분이 둘 이상이면 건너뛰고 연결된 20개를 모은다. 임베딩은 그중 앞 5개",
}


def md5_file(path, block=1 << 22):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(block), b""):
            h.update(chunk)
    return h.hexdigest()


def md5_bytes(b):
    return hashlib.md5(b).hexdigest()


def emit_bytes(path, data):
    """처음이면 쓰고, 이미 있으면 내용이 같은지 확인한다(고정 뒤에는 바뀌면 안 된다)."""
    path = Path(path)
    if path.exists():
        if path.read_bytes() != data:
            raise RuntimeError(f"{path.name}: 고정된 파일과 내용이 다르다. 의도한 변경이면 파일을 지우고 다시 실행한다")
        print(f"  검증: {path.name} (기존 파일과 같다)")
    else:
        path.write_bytes(data)
        print(f"  저장: {path.name} ({len(data):,} bytes)")
    return md5_bytes(data)


def emit_npy(path, arr):
    buf = io.BytesIO()
    np.save(buf, np.ascontiguousarray(arr), allow_pickle=False)
    return emit_bytes(path, buf.getvalue())


def emit_csv(path, df):
    return emit_bytes(path, df.to_csv(index=False, lineterminator="\n").encode("utf-8-sig"))


def emit_json(path, obj):
    return emit_bytes(path, json.dumps(el.to_jsonable(obj), ensure_ascii=False, indent=1, sort_keys=True).encode("utf-8"))


CHECKS = {}


def record(name, ok, **detail):
    """합성 검산 결과를 기록하고, 실패하면 멈춘다. detail에는 시간 같은 실행마다 달라지는 값을 넣지 않는다(s4_freeze.json에 들어간다)."""
    CHECKS[name] = {"ok": bool(ok), **{k: (round(v, 6) if isinstance(v, float) else v) for k, v in detail.items()}}
    shown = {k: (round(v, 6) if isinstance(v, float) else v) for k, v in detail.items()}
    print(("통과  " if ok else "실패  ") + name + ("  " + str(shown) if detail else ""))
    assert ok, f"합성 검산 실패: {name} {detail}"

C:\dev\kfm-spike\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


실행 시각: 2026-10-19T21:31:41 | python 3.11.9 | Windows-10-10.0.26200-SP0
실행 파일: C:\dev\kfm-spike\.venv\Scripts\python.exe | RUN_REAL = True | N_WORKERS = 10
torch 2.14.1+cpu | geoopt 0.5.1 | numpy 2.4.6 | scipy 1.17.1 | pandas 3.0.6 | numba 0.68.0 | 스레드(torch) 1
설정: {'dim': 2, 'init_halfwidth': 0.1, 'adam_lr': 0.05, 'adam_window': 200, 'adam_tol': 1e-06, 'adam_max_iter': 50000, 'r_max': 5.0, 'fin_ftol': 1e-12, 'fin_gtol': 1e-10, 'fin_maxiter': 20000, 'fin_maxfun': 50000, 'mds_max_norm': 0.14142135623730953, 'm1': 0.5, 'm2': 0.9, 'n_euc_u': 4, 'n_hyp_u': 3, 'retry_offsets': (5, 10), 'flat_rtol': 0.001, 'ratio_max': 0.8, 'n_worst': 20, 'tie_rtol': 1e-12, 'resample_frac': 0.2, 'resample_scale': 5}
config_hash: c2ae3329a5afc3d944bb26ba125c1ad4


## 1절. 합성 검산 (실데이터 아님)

판정 계산 코드를 합성 자료로 검산한다. 하나라도 실패하면 멈춘다.

### 1-1. tree(깊이 4) 재현: PLAN 7.1 초기값(두 공간 공통 U(±0.1)², 시드 0–4)

S1b(`00_tree_baseline.ipynb`) 값을 이 라이브러리로 다시 계산해 맞춘다: 쌍곡 최솟값 0.03394, 유클리드 최솟값 0.15063, 비 0.2253. 마무리 값은 ftol이 정한 정밀도 안에서만 정해지므로(왜곡 소수 6자리에서 ±5e-7) 시드별 허용 오차는 3e-6이다. 이 셀의 실행 시간은 3절 실행 시간 추정의 기준이다.

In [2]:
TREE_D = el.tree_hop_distances(4)
assert TREE_D.shape == (31, 31) and np.allclose(TREE_D, TREE_D.T) and TREE_D.max() == 8
S1B = {"hyp": [0.099059, 0.033939, 0.033939, 0.033939, 0.033939], "euc": [0.150633, 0.188722, 0.188722, 0.150633, 0.150633]}
rows, t_all = [], time.perf_counter()
for seed in range(5):
    X0 = el.init_uniform(31, seed, CFG)
    h = el.optimize_from_init(TREE_D, "poincare", X0, CFG)
    e = el.optimize_from_init(TREE_D, "euclid", X0, CFG)
    np_h, np_e = el.np_distortion(TREE_D, h["X"], "poincare")[0], el.np_distortion(TREE_D, e["X"], "euclid")[0]
    rows.append({"seed": seed, "hyp": h["distortion"], "hyp_conv": h["finish_converged"], "hyp_iter": f"{h['n_iter_adam']}+{h['n_iter_finish']}",
                 "hyp_cap": h["n_at_cap"], "hyp_maxr": h["max_radius"], "euc": e["distortion"], "euc_conv": e["finish_converged"],
                 "euc_iter": f"{e['n_iter_adam']}+{e['n_iter_finish']}", "np_diff": max(abs(np_h - h["distortion"]), abs(np_e - e["distortion"])),
                 "hyp_sec": h["seconds"], "euc_sec": e["seconds"], "hyp_adam_iter": h["n_iter_adam"], "euc_adam_iter": e["n_iter_adam"],
                 "hyp_fin_iter": h["n_iter_finish"], "euc_fin_iter": e["n_iter_finish"]})
TREE_SECONDS = time.perf_counter() - t_all
TREE = pd.DataFrame(rows).set_index("seed")
display(TREE[["hyp", "hyp_conv", "hyp_iter", "hyp_cap", "hyp_maxr", "euc", "euc_conv", "euc_iter", "hyp_sec", "euc_sec"]])
hyp_min, euc_min = TREE["hyp"].min(), TREE["euc"].min()
ratio_min = hyp_min / euc_min
d_h, d_e = np.abs(TREE["hyp"].to_numpy() - S1B["hyp"]).max(), np.abs(TREE["euc"].to_numpy() - S1B["euc"]).max()
print(f"쌍곡 최솟값 {hyp_min:.6f}, 유클리드 최솟값 {euc_min:.6f}, 비 {ratio_min:.4f}; S1b와 최대 차이 쌍곡 {d_h:.2e}, 유클리드 {d_e:.2e}; 실행 {TREE_SECONDS:.1f} s")
TREE_STATS = {
    "seconds_total": TREE_SECONDS,
    "hyp_sec_mean": float(TREE["hyp_sec"].mean()), "euc_sec_mean": float(TREE["euc_sec"].mean()),
    "hyp_adam_iter_mean": float(TREE["hyp_adam_iter"].mean()), "euc_adam_iter_mean": float(TREE["euc_adam_iter"].mean()),
    "hyp_fin_iter_mean": float(TREE["hyp_fin_iter"].mean()), "euc_fin_iter_mean": float(TREE["euc_fin_iter"].mean()),
    "n_pairs": 465,
}
record("tree: S1b 값 재현 (쌍곡 0.03394, 유클리드 0.15063, 비 0.2253)",
       d_h < 3e-6 and d_e < 3e-6 and abs(hyp_min - 0.03394) < 1e-5 and abs(euc_min - 0.15063) < 1e-5 and abs(ratio_min - 0.2253) < 5e-4
       and bool(TREE["hyp_conv"].all()) and bool(TREE["euc_conv"].all()),
       hyp_min=float(hyp_min), euc_min=float(euc_min), ratio=float(ratio_min), max_diff_vs_S1b=float(max(d_h, d_e)))
record("tree: torch/geoopt 왜곡 = 독립 numpy 구현", TREE["np_diff"].max() < 1e-9, max_diff=float(TREE["np_diff"].max()))

,hyp,hyp_conv,hyp_iter,hyp_cap,hyp_maxr,euc,euc_conv,euc_iter,hyp_sec,euc_sec
seed,,,,,,,,,,
0,0.099059,True,9405+710,8,5.0,0.150633,True,361+1,8.455621,0.064845
1,0.033938,True,10343+312,4,5.0,0.188722,True,374+1,7.790388,0.064513
2,0.033939,True,12098+668,4,5.0,0.188722,True,508+1,9.245507,0.089515
3,0.033938,True,11853+491,4,5.0,0.150633,True,410+1,9.058885,0.072160
4,0.033939,True,10652+501,4,5.0,0.150633,True,380+1,8.231568,0.065089


쌍곡 최솟값 0.033938, 유클리드 최솟값 0.150633, 비 0.2253; S1b와 최대 차이 쌍곡 5.15e-07, 유클리드 2.06e-07; 실행 43.1 s
통과  tree: S1b 값 재현 (쌍곡 0.03394, 유클리드 0.15063, 비 0.2253)  {'hyp_min': 0.033938, 'euc_min': 0.150633, 'ratio': 0.225306, 'max_diff_vs_S1b': 1e-06}
통과  tree: torch/geoopt 왜곡 = 독립 numpy 구현  {'max_diff': 0.0}


### 1-2. 재시작 규약(ADR 0013 4항) 전체 실행: tree에 시도 한 번

유클리드 5(고전 MDS + U 시드 0..3), 쌍곡 5(줄인 초기값 m₁ = 0.5, m₂ = 0.9 + U 시드 0..2)를 여러 프로세스로 돌린다(각 1스레드, spawn). 자식 프로세스 결과가 위 직렬 결과와 같은지(시드로 결정되는지)도 확인한다.

In [3]:
t0 = time.perf_counter()
with el.PoolRunner(min(N_WORKERS, 8)) as pool:
    TREE_EMB = el.embed_input(TREE_D, pool, CFG, name="tree")
PROTO_SECONDS = time.perf_counter() - t0
S = TREE_EMB["summary"]
rs = TREE_EMB["restarts"]
euc_ids = [(r["init"], r["seed"]) for r in rs if r["space"] == "euclid"]
hyp_ids = [(r["init"], r["seed"], r["m"]) for r in rs if r["space"] == "poincare"]
print("유클리드 재시작:", euc_ids)
print("쌍곡 재시작:", hyp_ids)
print(f"프로세스 풀 실행 {PROTO_SECONDS:.1f} s")
display(el.restart_table(TREE_EMB))
display(el.embedding_table({"tree": TREE_EMB}).T)
same = [r for r in rs if r["space"] == "poincare" and r["init"] == "U" and r["seed"] == 2][0]
serial_seed2 = el.optimize_from_init(TREE_D, "poincare", el.init_uniform(31, 2, CFG), CFG)
record("재시작 구성: 유클리드 MDS + U 0..3, 쌍곡 U 0..2 + 줄인 초기값 m1, m2",
       euc_ids == [("mds", None), ("U", 0), ("U", 1), ("U", 2), ("U", 3)]
       and hyp_ids == [("U", 0, None), ("U", 1, None), ("U", 2, None), ("shrunk_m1", None, M1), ("shrunk_m2", None, M2)])
record("자식 프로세스 결과 = 직렬 결과(같은 시드)", same["distortion"] == serial_seed2["distortion"] and np.array_equal(same["X"], serial_seed2["X"]))
record("tree 판정: 쌍곡 이점 있음, 비 0.2253, 수렴 5/5, 시도 1회",
       S["advantage"] == el.ADV_YES and abs(S["ratio"] - 0.2253) < 5e-4 and S["n_converged_hyp"] == 5 and S["n_converged_euc"] == 5
       and TREE_EMB["final_attempt"] == 0 and S["hyp_best"]["n_at_cap"] == 4,
       ratio=float(S["ratio"]), stress_ratio=float(S["stress_ratio"]), n_at_cap=int(S["hyp_best"]["n_at_cap"]))

유클리드 재시작: [('mds', None), ('U', 0), ('U', 1), ('U', 2), ('U', 3)]
쌍곡 재시작: [('U', 0, None), ('U', 1, None), ('U', 2, None), ('shrunk_m1', None, 0.5), ('shrunk_m2', None, 0.9)]
프로세스 풀 실행 19.7 s


,attempt,space,init,seed,m,stress,distortion,finish_converged,adam_converged,n_iter_adam,n_iter_finish,n_at_cap,max_radius,seconds
0,0,euclid,mds,NaN,NaN,18.023729,0.150633,True,True,338,1,NaN,NaN,1.616613
1,0,euclid,U,0.0,NaN,18.023729,0.150633,True,True,361,1,NaN,NaN,1.598591
2,0,euclid,U,1.0,NaN,26.271236,0.188722,True,True,374,1,NaN,NaN,1.617868
3,0,euclid,U,2.0,NaN,26.271236,0.188722,True,True,508,1,NaN,NaN,1.691054
4,0,euclid,U,3.0,NaN,18.023729,0.150633,True,True,410,1,NaN,NaN,1.609648
5,0,poincare,U,0.0,NaN,8.022814,0.099059,True,True,9405,710,8.0,5.0,9.615086
6,0,poincare,U,1.0,NaN,1.042767,0.033938,True,True,10343,312,4.0,5.0,10.258232
7,0,poincare,U,2.0,NaN,1.042767,0.033939,True,True,12098,668,4.0,5.0,11.929605
8,0,poincare,shrunk_m1,NaN,0.5,1.042767,0.033939,True,True,5124,271,4.0,5.0,3.943172
9,0,poincare,shrunk_m2,NaN,0.9,1.042767,0.033939,True,True,5521,256,4.0,5.0,4.206787


,0
입력,tree
n,31
기록,쌍곡 이점 있음
쌍곡 이점,쌍곡 이점 있음
평균 상대 왜곡 쌍곡,0.033939
평균 상대 왜곡 유클리드,0.150633
왜곡 비,0.225306
stress 비(판정 재시작),0.057855
stress 비(전체 최솟값),0.057855
수렴 쌍곡/유클리드,"5/5, 5/5"


통과  재시작 구성: 유클리드 MDS + U 0..3, 쌍곡 U 0..2 + 줄인 초기값 m1, m2
통과  자식 프로세스 결과 = 직렬 결과(같은 시드)
통과  tree 판정: 쌍곡 이점 있음, 비 0.2253, 수렴 5/5, 시도 1회  {'ratio': 0.225306, 'stress_ratio': 0.057855, 'n_at_cap': 4}


### 1-3. δ fixture

`tree`는 δ = δ̄ = 0, `c4`는 δ_rel = 1, `square4`는 δ_rel = 2 − √2. 무작위 거리에 지역별 상수 g_i + g_j를 더해도 δ와 δ̄가 같다. numba 구현(서로 다른 네 점 전체, 기준점 (max, min) 행렬곱)은 느린 파이썬 반복과도 맞춰 본다.

In [4]:
rep_tree, rep_c4, rep_sq = el.delta_report(TREE_D), el.delta_report(el.c4_distances()), el.delta_report(el.square4_distances())
record("δ fixture tree: δ = δ̄ = 0", rep_tree["delta"] == 0 and rep_tree["delta_bar"] == 0 and rep_tree["delta_quad"] == 0, n_quads=rep_tree["n_quads"], n_ties=rep_tree["n_ties"])
record("δ fixture c4: δ_rel = 1", abs(rep_c4["delta_rel"] - 1.0) < 1e-12, delta_rel=rep_c4["delta_rel"], delta_bar=rep_c4["delta_bar"])
record("δ fixture square4: δ_rel = 2 - sqrt(2)", abs(rep_sq["delta_rel"] - (2 - math.sqrt(2))) < 1e-12, delta_rel=rep_sq["delta_rel"], delta_bar=rep_sq["delta_bar"])


def rand_sym(n, seed):
    r = np.random.default_rng(seed).uniform(1.0, 10.0, (n, n))
    r = (r + r.T) / 2.0
    np.fill_diagonal(r, 0.0)
    return r


rng = np.random.default_rng(20261019)
DM = el.random_metric(40, seed=7)
g = rng.uniform(0.0, 3.0, 40)
DG = DM + g[:, None] + g[None, :]
np.fill_diagonal(DG, 0.0)
a, b = el.delta_report(DM), el.delta_report(DG)
record("δ·δ̄ 불변: 무작위 거리(삼각 부등식 성립)에 g_i + g_j",
       abs(a["delta"] - b["delta"]) < 1e-10 * a["delta"] and abs(a["delta_bar"] - b["delta_bar"]) < 1e-12 and a["delta_diff"] < 1e-10 and b["delta_diff"] < 1e-10,
       delta=a["delta"], delta_g=b["delta"], delta_bar=a["delta_bar"], delta_bar_g=b["delta_bar"])
M = rand_sym(30, 5)
MG = M + g[:30, None] + g[None, :30]
np.fill_diagonal(MG, 0.0)
sa, sb = el.four_point_stats(M), el.four_point_stats(MG)
record("δ·δ̄ 불변: 임의의 대칭 거리(삼각 부등식 없음)에 g_i + g_j", abs(sa["delta"] - sb["delta"]) < 1e-10 and abs(sa["delta_bar"] - sb["delta_bar"]) < 1e-12,
       delta=sa["delta"], delta_g=sb["delta"], delta_bar=sa["delta_bar"], delta_bar_g=sb["delta_bar"])


def brute_force(D, tie_rtol=CFG["tie_rtol"]):
    n = len(D)
    tot, mx, ties, cnt = 0.0, 0.0, 0, 0
    for i, j, k, l in itertools.combinations(range(n), 4):
        s = sorted([D[i, j] + D[k, l], D[i, k] + D[j, l], D[i, l] + D[j, k]], reverse=True)
        cnt += 1
        mx = max(mx, (s[0] - s[1]) / 2)
        if s[0] - s[2] <= tie_rtol * s[0]:
            ties += 1
        else:
            tot += (s[0] - s[1]) / (s[0] - s[2])
    return {"delta": mx, "delta_bar": tot / cnt, "n_quads": cnt, "n_ties": ties}


for label, D_ in (("메트릭 n=14", el.random_metric(14, 3)), ("임의 대칭 n=13", rand_sym(13, 5)),
                  ("tree 깊이 3 n=15", el.tree_hop_distances(3)), ("c4", el.c4_distances())):
    bf, nb = brute_force(D_), el.four_point_stats(D_)
    ok = abs(bf["delta"] - nb["delta"]) < 1e-12 and abs(bf["delta_bar"] - nb["delta_bar"]) < 1e-12 and bf["n_quads"] == nb["n_quads"] and bf["n_ties"] == nb["n_ties"]
    record(f"numba = 파이썬 반복 ({label})", ok, delta=nb["delta"], delta_bar=nb["delta_bar"])
for n_, seed_ in ((30, 2), (60, 3)):
    D_ = el.random_metric(n_, seed_)
    record(f"기준점 (max, min) 행렬곱 δ = 네 점 전체 δ (메트릭 n={n_})", abs(el.delta_refpoint(D_) - el.four_point_stats(D_)["delta"]) < 1e-12)
try:
    el.delta_refpoint(M)
    rejected = False
except ValueError:
    rejected = True
record("기준점 방식은 삼각 부등식이 깨진 거리를 거부한다", rejected)
t0 = time.perf_counter()
rep250 = el.delta_report(el.random_metric(250, 11))
print(f"n = 250 무작위 메트릭 δ·δ̄ 전체 계산 {time.perf_counter() - t0:.2f} s (네 점 {rep250['n_quads']:,}개)")

통과  δ fixture tree: δ = δ̄ = 0  {'n_quads': 31465, 'n_ties': 2656}
통과  δ fixture c4: δ_rel = 1  {'delta_rel': 1.0, 'delta_bar': 1.0}
통과  δ fixture square4: δ_rel = 2 - sqrt(2)  {'delta_rel': 0.585786, 'delta_bar': 1.0}
통과  δ·δ̄ 불변: 무작위 거리(삼각 부등식 성립)에 g_i + g_j  {'delta': 0.996915, 'delta_g': 0.996915, 'delta_bar': 0.348508, 'delta_bar_g': 0.348508}
통과  δ·δ̄ 불변: 임의의 대칭 거리(삼각 부등식 없음)에 g_i + g_j  {'delta': 6.228914, 'delta_g': 6.228914, 'delta_bar': 0.501392, 'delta_bar_g': 0.501392}
통과  numba = 파이썬 반복 (메트릭 n=14)  {'delta': 1.068622, 'delta_bar': 0.345256}
통과  numba = 파이썬 반복 (임의 대칭 n=13)  {'delta': 4.863044, 'delta_bar': 0.501896}
통과  numba = 파이썬 반복 (tree 깊이 3 n=15)  {'delta': 0.0, 'delta_bar': 0.0}
통과  numba = 파이썬 반복 (c4)  {'delta': 1.0, 'delta_bar': 1.0}
통과  기준점 (max, min) 행렬곱 δ = 네 점 전체 δ (메트릭 n=30)
통과  기준점 (max, min) 행렬곱 δ = 네 점 전체 δ (메트릭 n=60)
통과  기준점 방식은 삼각 부등식이 깨진 거리를 거부한다


n = 250 무작위 메트릭 δ·δ̄ 전체 계산 0.47 s (네 점 158,882,750개)


### 1-4. `gravity_grid`, `d_flow`

격자 지역 + 지수 중력 OD: ℓ ≥ 0, 폐포 후 삼각 부등식, 폐포 전 ℓ_sym의 δ(서로 다른 네 점) = β·δ(`d_sym`)이고 δ̄가 같다. 대안 정규화 ℓ′는 ADR 0010 2항의 관계식 ℓ′ = ℓ_sym + h_i + h_j − log 2 − log(AM_ij/GM_ij)와 맞춘다. 폐포는 독립 구현(scipy `shortest_path`)과 희소 OD에서 맞춘다.

In [5]:
GG = el.gravity_grid(k=7, beta=0.08, seed=1)
T_gg, d_gg, beta_gg = GG["T"], GG["d_sym"], GG["beta"]
n_gg = len(T_gg)
off_gg = ~np.eye(n_gg, dtype=bool)
FD = el.flow_distance(T_gg, kind="estimated", role=None)
record("gravity_grid: ℓ_sym, ℓ′ >= 0", FD["raw_main"][off_gg].min() >= 0 and FD["raw_alt"][off_gg].min() >= 0,
       min_main=float(FD["raw_main"][off_gg].min()), min_alt=float(FD["raw_alt"][off_gg].min()))
record("gravity_grid: 폐포 후 삼각 부등식", el.triangle_violation(FD["d_main"]) <= 1e-12 and el.triangle_violation(FD["d_alt"]) <= 1e-12,
       violation_main=el.triangle_violation(FD["d_main"]), violation_alt=el.triangle_violation(FD["d_alt"]))
raw0 = np.where(off_gg, FD["raw_main"], 0.0)
s_raw, s_d = el.four_point_stats(raw0), el.four_point_stats(d_gg)
record("gravity_grid: 폐포 전 ℓ_sym의 δ = β·δ(d_sym), δ̄ 같음",
       abs(s_raw["delta"] - beta_gg * s_d["delta"]) < 1e-10 * s_raw["delta"] and abs(s_raw["delta_bar"] - s_d["delta_bar"]) < 1e-12,
       delta_raw=s_raw["delta"], beta_delta=beta_gg * s_d["delta"], delta_bar_raw=s_raw["delta_bar"], delta_bar_d=s_d["delta_bar"])
T0_gg, O_gg, D_gg, V_gg = el.od_margins(T_gg)
h_gg = 0.5 * np.log(V_gg) - 0.25 * np.log(O_gg * D_gg)
AM, GM = (T0_gg + T0_gg.T) / 2, np.sqrt(T0_gg * T0_gg.T)
with np.errstate(all="ignore"):
    pred = FD["raw_main"] + h_gg[:, None] + h_gg[None, :] - math.log(2) - np.log(AM / GM)
err_alt = float(np.abs(pred[off_gg] - FD["raw_alt"][off_gg]).max())
record("ℓ′ = ℓ_sym + h_i + h_j - log 2 - log(AM/GM) (ADR 0010 2항 관계식)", err_alt < 1e-12, max_abs_err=err_alt)

# 폐포가 실제로 일을 하는 희소 OD: 독립 구현(scipy)과 맞춘다
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import shortest_path

for seed_ in range(100):
    rr = np.random.default_rng(seed_)
    n_s = 40
    Ts = rr.lognormal(2.0, 1.5, (n_s, n_s)) * (rr.random((n_s, n_s)) < 0.35)
    np.fill_diagonal(Ts, 0.0)
    try:
        FDs = el.flow_distance(Ts, kind="observed", role="observed")
        break
    except el.DisconnectedFlowError:
        continue
edge_s = FDs["edge"]
ref = shortest_path(csr_matrix(np.where(edge_s, FDs["raw_main"], 0.0)), directed=False)
ref_alt = shortest_path(csr_matrix(np.where(edge_s, FDs["raw_alt"], 0.0)), directed=False)
shorter = int(((FDs["d_main"] < FDs["raw_main"] - 1e-12) & edge_s).sum() // 2)
record("폐포 = scipy shortest_path (희소 OD, 폐포로 줄어든 엣지가 있음)",
       np.abs(FDs["d_main"] - ref).max() < 1e-10 and np.abs(FDs["d_alt"] - ref_alt).max() < 1e-10 and shorter > 0
       and el.triangle_violation(FDs["d_main"]) <= 1e-12 and np.allclose(FDs["d_main"], FDs["d_main"].T),
       seed=seed_, edges=int(edge_s.sum() // 2), shortened_edges=shorter)

통과  gravity_grid: ℓ_sym, ℓ′ >= 0  {'min_main': 1.689488, 'min_alt': 1.582789}
통과  gravity_grid: 폐포 후 삼각 부등식  {'violation_main': 0.0, 'violation_alt': 0.0}
통과  gravity_grid: 폐포 전 ℓ_sym의 δ = β·δ(d_sym), δ̄ 같음  {'delta_raw': 1.988225, 'beta_delta': 1.988225, 'delta_bar_raw': 0.324409, 'delta_bar_d': 0.324409}
통과  ℓ′ = ℓ_sym + h_i + h_j - log 2 - log(AM/GM) (ADR 0010 2항 관계식)  {'max_abs_err': 0.0}
통과  폐포 = scipy shortest_path (희소 OD, 폐포로 줄어든 엣지가 있음)  {'seed': 0, 'edges': 89, 'shortened_edges': 1}


### 1-5. 추정 OD 거부, 한 방향 0, 연결 성분, 제외 지역

관측 모드(`role = "observed"`)는 추정 OD를 거부하고, 대조군 모드(`role = None`)는 받는다. 한 방향이라도 0이면 엣지가 없고, 연결 성분이 둘 이상이면 오류다. `flow_exclusion`은 쌍방향 그래프의 본토 성분 밖에 있는 지역을 섬 연결 지역일 때만 빼고, 그 밖의 이유로 끊기면 중단한다.

In [6]:
try:
    el.flow_distance(T_gg, kind="estimated", role="observed")
    rejected = False
except el.EstimatedODError as err:
    rejected = True
    print("관측 모드 거부 메시지:", err)
el.flow_distance(T_gg, kind="observed", role="observed")
el.flow_distance(T_gg, kind="estimated", role=None)
record("관측 모드는 추정 OD를 거부, 대조군 모드(role=None)는 받음", rejected)

T_one = T_gg.copy()
T_one[0, 1] = 0.0
m_one = el.flow_edge_mask(T_one)
record("한 방향이라도 0이면 엣지 없음", (not m_one[0, 1]) and (not m_one[1, 0]) and el.flow_edge_mask(T_gg)[0, 1])

T_cut = T_gg.copy()
T_cut[:3, 3:] = 0.0
try:
    el.flow_distance(T_cut, kind="observed", role="observed")
    raised = False
except el.DisconnectedFlowError as err:
    raised = True
    print("연결 성분 오류 메시지:", err)
record("연결 성분이 둘 이상이면 오류", raised)

# flow_exclusion: 섬 연결 지역(마스크)이 본토와 쌍방향 통행이 없으면 빼고, 아니면 중단
isl = np.zeros(n_gg, dtype=bool)
isl[:3] = True
fx = el.flow_exclusion(T_cut, isl)
T_cut_keep = T_cut[np.ix_(fx["keep"], fx["keep"])]
el.flow_distance(T_cut_keep, kind="observed", role="observed")
try:
    el.flow_exclusion(T_cut, np.zeros(n_gg, dtype=bool))
    stopped = False
except el.DisconnectedFlowError:
    stopped = True
T_zero = T_gg.copy()
T_zero[5, :] = 0.0
T_zero[:, 5] = 0.0
fx0 = el.flow_exclusion(T_zero, np.zeros(n_gg, dtype=bool))
record("flow_exclusion: 섬 연결 지역만 제외, 그 밖은 중단, 통행이 전혀 없는 지역은 목록으로",
       fx["island_dropped"].tolist() == [0, 1, 2] and stopped and fx0["unlinked"].tolist() == [5] and not fx0["keep"][5])

관측 모드 거부 메시지: 추정 OD(중력모형 등)는 관측 입력으로 받지 않는다. 대조군은 role=None으로 부른다(ADR 0010 2항)
통과  관측 모드는 추정 OD를 거부, 대조군 모드(role=None)는 받음
통과  한 방향이라도 0이면 엣지 없음
연결 성분 오류 메시지: 연결 성분 2개(크기 [3, 46]). 빠진 지역은 미리 빼야 한다
통과  연결 성분이 둘 이상이면 오류
통과  flow_exclusion: 섬 연결 지역만 제외, 그 밖은 중단, 통행이 전혀 없는 지역은 목록으로


### 1-6. 중력 대조군: 적합, 재표본, 건너뛴 시드

이중 제약 지수 모형의 β를 평균 통행시간(방향 통행시간, 관측 T로 가중)에 맞추고 Furness로 균형을 잡는다. `gravity_grid`와 방향 통행시간이 비대칭인 OD에서 알려진 β가 복원되는지, 재표본이 명세대로(총량, ×5, 관측 0 패턴, 다른 난수열)인지, 연결 성분이 둘 이상인 시드를 건너뛰는지 본다.

In [7]:
fit = el.fit_gravity(T_gg, d_gg)
record("fit_gravity: 알려진 β 복원 (대칭 통행시간)", abs(fit["beta"] - beta_gg) < 1e-10 and np.abs(fit["T_exp"] - T0_gg).max() / T0_gg.max() < 1e-9,
       beta=fit["beta"], furness_iter=fit["furness_iter"], mean_time=fit["mean_time_obs"])
rr = np.random.default_rng(3)
t_asym = d_gg * (1 + 0.2 * rr.uniform(-1, 1, d_gg.shape))
np.fill_diagonal(t_asym, 0.0)
a_, b_ = np.exp(rr.normal(0, 1, n_gg)), np.exp(rr.normal(0, 1, n_gg))
T_as = 500.0 * a_[:, None] * b_[None, :] * np.exp(-0.06 * t_asym)
np.fill_diagonal(T_as, 0.0)
fit_as = el.fit_gravity(T_as, t_asym)
record("fit_gravity: 알려진 β 복원 (비대칭 방향 통행시간)", abs(fit_as["beta"] - 0.06) < 1e-10 and fit_as["row_sum_max_rel_err"] < 1e-10 and fit_as["col_sum_max_rel_err"] < 1e-10,
       beta=fit_as["beta"], mean_time_obs=fit_as["mean_time_obs"], mean_time_fit=fit_as["mean_time_fit"])

Tc, n_eff = el.resample_control(fit["T_exp"], T_gg, seed=3, cfg=CFG)
Tc_other, _ = el.resample_control(fit["T_exp"], T_gg, seed=4, cfg=CFG)
off_sum = float(T0_gg.sum())
rng_resample = np.random.default_rng([el.RESAMPLE_STREAM, 3]).random(3)
rng_embed = np.random.default_rng(3).random(3)
record("재표본: 총량 = round(0.2 x 존 간 합) x 5, 5의 배수, 대각 0, 시드마다 다름, 같은 시드는 같음, 임베딩과 다른 난수열",
       n_eff == int(math.floor(0.2 * off_sum + 0.5)) and Tc.sum() == 5 * n_eff and np.all(Tc % 5 == 0) and np.all(np.diag(Tc) == 0)
       and not np.array_equal(Tc, Tc_other) and not np.allclose(rng_resample, rng_embed)
       and np.array_equal(Tc, el.resample_control(fit["T_exp"], T_gg, seed=3, cfg=CFG)[0]),
       n_eff=int(n_eff), total=float(Tc.sum()))
T_obs0 = T_gg.copy()
T_obs0[rr.random(T_obs0.shape) < 0.3] = 0.0
np.fill_diagonal(T_obs0, 0.0)
Tc0, n_eff0 = el.resample_control(fit["T_exp"], T_obs0, seed=5, cfg=CFG)
record("재표본: 관측의 0 패턴 적용 (총량은 0 패턴 때문에 5 x 총량보다 작다)", bool(np.all(Tc0[T_obs0 == 0] == 0)) and Tc0.sum() < 5 * n_eff0)

# 건너뛴 시드: 지역 0의 통행량을 줄여 표본에서 끊기는 시드가 생기게 한다
T_thin = T_gg.copy()
scale0 = 15.0 / O_gg[0]
T_thin[0, :] *= scale0
T_thin[:, 0] *= scale0
np.fill_diagonal(T_thin, 0.0)
fit_thin = el.fit_gravity(T_thin, d_gg)
ctrls, skipped = el.make_controls(fit_thin["T_exp"], T_thin, n_wanted=6, cfg=CFG)
seeds_all = sorted([c["seed"] for c in ctrls] + [s["seed"] for s in skipped])
record("make_controls: 연결 성분이 둘 이상인 시드를 건너뛰고 연결된 대조군만 모은다",
       len(skipped) > 0 and len(ctrls) == 6 and seeds_all == list(range(seeds_all[-1] + 1)) and [c["seed"] for c in ctrls] == sorted(c["seed"] for c in ctrls)
       and all(el.flow_components(c["T"])[0] == 1 for c in ctrls),
       control_seeds=[c["seed"] for c in ctrls], skipped_seeds=[s["seed"] for s in skipped])

통과  fit_gravity: 알려진 β 복원 (대칭 통행시간)  {'beta': 0.08, 'furness_iter': 29, 'mean_time': 20.46864}
통과  fit_gravity: 알려진 β 복원 (비대칭 방향 통행시간)  {'beta': 0.06, 'mean_time_obs': 22.434208, 'mean_time_fit': 22.434208}
통과  재표본: 총량 = round(0.2 x 존 간 합) x 5, 5의 배수, 대각 0, 시드마다 다름, 같은 시드는 같음, 임베딩과 다른 난수열  {'n_eff': 90864, 'total': 454320.0}
통과  재표본: 관측의 0 패턴 적용 (총량은 0 패턴 때문에 5 x 총량보다 작다)
통과  make_controls: 연결 성분이 둘 이상인 시드를 건너뛰고 연결된 대조군만 모은다  {'control_seeds': [0, 1, 5, 7, 8, 10], 'skipped_seeds': [2, 3, 4, 6, 9]}


### 1-7. 판정 함수 대표 사례와 재실행 규칙

실제 최적화 없이 미리 정한 stress를 돌려주는 가짜 runner로 재실행 규칙(쌍곡 U 재시작 3개만 시드 +5, +10으로 다시 돌림, 유클리드와 줄인 초기값은 그대로, 쌍곡은 모든 시도를 합쳐 판정, 시도마다 합친 쌍곡 최솟값으로 실패 판정, 두 번 재실행해도 실패이면 "최적화 실패"), 평탄 극한, 보류를 확인한다. 이어서 `d_flow` 쌍곡 이점(대조군 비교)과 중력모형 이상의 위계 판정의 대표 사례를 확인한다.

In [8]:
class FakeRunner:
    """실제 최적화 없이 (공간, 초기값 종류, 시드)별로 정한 stress를 돌려주는 가짜 runner(재실행 규칙 점검용)."""

    def __init__(self, n, rule):
        self.known = {el.init_uniform(n, s, CFG).tobytes(): s for s in range(40)}
        self.rule, self.calls = rule, []

    def run(self, tasks):
        out = []
        for t in tasks:
            seed = self.known.get(np.ascontiguousarray(t["X0"], dtype=float).tobytes())
            kind = "U" if seed is not None else "other"
            self.calls.append((t["space"], kind, seed))
            stress, conv = self.rule(t["space"], kind, seed)
            out.append({"space": t["space"], "stress": stress, "distortion": stress / 10.0, "finish_converged": conv, "adam_converged": True,
                        "n_iter_adam": 1, "n_iter_finish": 1, "n_iter": 2, "seconds": 0.0, "X": t["X0"], "max_radius": 1.0, "n_at_cap": 0})
        return out


D15 = el.tree_hop_distances(3)


def fake_embed(rule):
    fr = FakeRunner(len(D15), rule)
    return el.embed_input(D15, fr, CFG, name="fake"), fr


def seeds_by_attempt(res, space):
    """시도마다 새로 돌린 U 재시작의 시드. 재실행 시도에는 쌍곡 U 재시작만 있다."""
    return [[r["seed"] for r in a["restarts"] if r["space"] == space and r["init"] == "U"] for a in res["attempts"]]


def n_calls(fr, space, kind=None):
    return sum(1 for c in fr.calls if c[0] == space and (kind is None or c[1] == kind))


# A) 첫 시도 실패(쌍곡 stress 12 > 유클리드 10), 쌍곡 U 시드 5..7로 다시 돌려 7이 나와 성공
res_a, fr_a = fake_embed(lambda sp, k, sd: (10.0, True) if sp == "euclid" else ((12.0, True) if (sd is None or sd < 5) else (7.0, True)))
record("재실행: 실패 -> 쌍곡 U 재시작 3개만 시드 +5로 다시 돌린다(유클리드 5, 줄인 초기값 2는 그대로), 합친 쌍곡 8개로 판정",
       res_a["final_attempt"] == 1 and seeds_by_attempt(res_a, "poincare") == [[0, 1, 2], [5, 6, 7]] and seeds_by_attempt(res_a, "euclid") == [[0, 1, 2, 3], []]
       and n_calls(fr_a, "euclid") == 5 and n_calls(fr_a, "poincare") == 8 and n_calls(fr_a, "poincare", "other") == 2
       and res_a["summary"]["n_restarts_hyp"] == 8 and res_a["summary"]["advantage"] == el.ADV_YES and abs(res_a["summary"]["ratio"] - 0.7) < 1e-12)
# B) 두 번 재실행해도 실패
res_b, fr_b = fake_embed(lambda sp, k, sd: (10.0, True) if sp == "euclid" else (12.0, True))
record("재실행: 두 번 재실행해도 실패 -> 기록 '최적화 실패', 쌍곡 이점 '보류', 쌍곡 U 시드 +5, +10, 합친 쌍곡 11개",
       len(res_b["attempts"]) == 3 and res_b["summary"]["record"] == el.REC_FAIL and res_b["summary"]["advantage"] == el.ADV_HOLD
       and seeds_by_attempt(res_b, "poincare") == [[0, 1, 2], [5, 6, 7], [10, 11, 12]] and res_b["summary"]["n_restarts_hyp"] == 11 and n_calls(fr_b, "euclid") == 5)
# C) 평탄 극한: 쌍곡 최솟값이 유클리드의 1 ~ 1.001배, 수렴 여부와 무관
res_c, _ = fake_embed(lambda sp, k, sd: (10.0, True) if sp == "euclid" else (10.005, False))
record("평탄 극한: 쌍곡 최솟값이 유클리드 x (1 + 1e-3) 이내이면 '쌍곡 이점 없음(평탄 극한)', 재실행 없음, 수렴 불필요",
       res_c["summary"]["record"] == el.ADV_FLAT and len(res_c["attempts"]) == 1 and not res_c["summary"]["opt_failure"])
# D) 쌍곡에 마무리가 수렴한 재시작이 없으면 보류
res_d, _ = fake_embed(lambda sp, k, sd: (10.0, True) if sp == "euclid" else (5.0, False))
record("보류: 한 공간에 마무리가 수렴한 재시작이 없다", res_d["summary"]["advantage"] == el.ADV_HOLD and res_d["summary"]["ratio"] is None)
# E) 비 0.5 / 0.9 와 판정 재시작 선택(수렴한 것 중 최소 stress)
res_e, _ = fake_embed(lambda sp, k, sd: (10.0, True) if sp == "euclid" else (5.0, True))
res_e2, _ = fake_embed(lambda sp, k, sd: (10.0, True) if sp == "euclid" else (9.0, True))
res_f, _ = fake_embed(lambda sp, k, sd: (10.0, True) if sp == "euclid" else ((4.0, False) if k == "U" else (6.0, True)))
record("쌍곡 이점 문턱 0.8: 비 0.5 있음, 비 0.9 없음, 수렴하지 않은 더 작은 stress는 판정에 쓰지 않음",
       res_e["summary"]["advantage"] == el.ADV_YES and res_e2["summary"]["advantage"] == el.ADV_NO and res_f["summary"]["hyp_best"]["stress"] == 6.0
       and abs(res_f["summary"]["ratio"] - 0.6) < 1e-12)


# G) 쌍곡은 모든 시도의 재시작을 합쳐 판정: 첫 시도의 수렴한 재시작(11)이, 재실행의 수렴하지 않은 더 작은 값(9)이 아니라 판정 재시작이 된다
res_g, _ = fake_embed(lambda sp, k, sd: (10.0, True) if sp == "euclid" else ((9.0, False) if sd == 5 else ((12.0, True) if sd in (6, 7) else (11.0, True))))
sg = res_g["summary"]
record("쌍곡은 모든 시도의 재시작을 합쳐 판정: 첫 시도 실패(11 > 10.01), 합친 최솟값 9 -> 성공, 판정 재시작은 첫 시도의 수렴한 11",
       res_g["final_attempt"] == 1 and not sg["opt_failure"] and sg["hyp_min_stress_all"] == 9.0 and sg["hyp_best"]["stress"] == 11.0 and sg["hyp_best"]["attempt"] == 0
       and sg["advantage"] == el.ADV_NO and abs(sg["ratio"] - 1.1) < 1e-12 and sg["n_restarts_hyp"] == 8)
# H) 실패 판정은 시도마다 합친 쌍곡 최솟값: 12 -> 10.5(아직 실패) -> 10.005(성공, 평탄 극한)
res_h, _ = fake_embed(lambda sp, k, sd: (10.0, True) if sp == "euclid" else ((10.005, False) if sd == 10 else ((10.5, True) if sd == 5 else (12.0, True))))
record("실패 판정은 시도마다 합친 쌍곡 최솟값으로: 12 -> 10.5 (실패) -> 10.005 (성공, 평탄 극한)",
       res_h["final_attempt"] == 2 and [a["summary"]["opt_failure"] for a in res_h["attempts"]] == [True, True, False]
       and res_h["summary"]["record"] == el.ADV_FLAT and res_h["summary"]["n_restarts_hyp"] == 11)


def S_(record_, adv, ratio):
    return {"record": record_, "advantage": adv, "ratio": ratio, "reason": "x"}


ctrl5 = [S_(el.ADV_NO, el.ADV_NO, r) for r in (0.95, 0.90, 0.97, 0.99, 0.92)]
jf = lambda flow, ctrl: el.judge_flow_advantage(flow, ctrl, CFG)["advantage"]  # noqa: E731
record("d_flow 쌍곡 이점: 비 <= 0.8이고 대조군 비의 최솟값보다 작아야 한다",
       jf(S_(el.ADV_YES, el.ADV_YES, 0.5), ctrl5) == el.ADV_YES
       and jf(S_(el.ADV_YES, el.ADV_YES, 0.7), [S_(el.ADV_YES, el.ADV_YES, 0.6)] + ctrl5[1:]) == el.ADV_NO
       and jf(S_(el.ADV_NO, el.ADV_NO, 0.85), ctrl5) == el.ADV_NO
       and jf(S_(el.ADV_FLAT, el.ADV_FLAT, 0.99), ctrl5) == el.ADV_FLAT)
record("d_flow 쌍곡 이점: 대조군 중 하나라도 판정 재시작이 없으면 보류",
       jf(S_(el.ADV_YES, el.ADV_YES, 0.5), ctrl5[:4] + [S_(el.REC_FAIL, el.ADV_HOLD, None)]) == el.ADV_HOLD
       and jf(S_(el.ADV_YES, el.ADV_YES, 0.5), ctrl5[:4] + [S_(el.ADV_HOLD, el.ADV_HOLD, None)]) == el.ADV_HOLD
       and jf(S_(el.ADV_HOLD, el.ADV_HOLD, None), ctrl5) == el.ADV_HOLD)

rngh = np.random.default_rng(1)
ctrl_db = {(nm, sub): 0.30 + 0.05 * rngh.random(20) for nm in el.NORMS for sub in el.SUBSETS}
flow_ok = {k: 0.25 for k in ctrl_db}
flow_one_bad = {**flow_ok, ("alt", "섬 제외"): float(ctrl_db[("alt", "섬 제외")].min() + 1e-6)}
flow_tie = {**flow_ok, ("main", "전체"): float(ctrl_db[("main", "전체")].min())}
record("중력모형 이상의 위계: 4개 조건 모두 대조군 20개보다 작아야 한다(하나라도 같거나 크면 아님)",
       el.judge_hierarchy(flow_ok, ctrl_db)["hierarchy"] and not el.judge_hierarchy(flow_one_bad, ctrl_db)["hierarchy"] and not el.judge_hierarchy(flow_tie, ctrl_db)["hierarchy"])
try:
    el.judge_hierarchy(flow_ok, {k: v[:19] for k, v in ctrl_db.items()})
    short_ok = False
except ValueError:
    short_ok = True
record("중력모형 이상의 위계: 대조군이 20개가 아니면 오류", short_ok)

통과  재실행: 실패 -> 쌍곡 U 재시작 3개만 시드 +5로 다시 돌린다(유클리드 5, 줄인 초기값 2는 그대로), 합친 쌍곡 8개로 판정
통과  재실행: 두 번 재실행해도 실패 -> 기록 '최적화 실패', 쌍곡 이점 '보류', 쌍곡 U 시드 +5, +10, 합친 쌍곡 11개
통과  평탄 극한: 쌍곡 최솟값이 유클리드 x (1 + 1e-3) 이내이면 '쌍곡 이점 없음(평탄 극한)', 재실행 없음, 수렴 불필요
통과  보류: 한 공간에 마무리가 수렴한 재시작이 없다
통과  쌍곡 이점 문턱 0.8: 비 0.5 있음, 비 0.9 없음, 수렴하지 않은 더 작은 stress는 판정에 쓰지 않음
통과  쌍곡은 모든 시도의 재시작을 합쳐 판정: 첫 시도 실패(11 > 10.01), 합친 최솟값 9 -> 성공, 판정 재시작은 첫 시도의 수렴한 11
통과  실패 판정은 시도마다 합친 쌍곡 최솟값으로: 12 -> 10.5 (실패) -> 10.005 (성공, 평탄 극한)
통과  d_flow 쌍곡 이점: 비 <= 0.8이고 대조군 비의 최솟값보다 작아야 한다
통과  d_flow 쌍곡 이점: 대조군 중 하나라도 판정 재시작이 없으면 보류
통과  중력모형 이상의 위계: 4개 조건 모두 대조군 20개보다 작아야 한다(하나라도 같거나 크면 아님)
통과  중력모형 이상의 위계: 대조군이 20개가 아니면 오류


### 1-8. 3절 배선 점검(합성 장난감 자료)

3절이 부르는 `embed_lib` 함수들을 합성 자료(5 × 5 격자 지역, 섬 연결 지역 2개, 포아송 잡음 OD)로 한 번 돌려 δ 표, 대조군 δ̄, 임베딩, 판정, 저장이 끝까지 되는지 본다. 값은 아무 의미도 없다(반복 수를 줄인 설정).

In [9]:
TOY = el.make_toy_data(k=5, seed=0, n_ctrl=6)
CFG_TOY = {**CFG, "adam_max_iter": 1500, "fin_maxiter": 1500, "fin_maxfun": 3000}
t0 = time.perf_counter()
with tempfile.TemporaryDirectory() as tmp, el.PoolRunner(min(N_WORKERS, 8), cache_dir=Path(tmp) / "cache") as pool_toy:
    TOY_RES = el.run_section3(TOY, pool_toy, CFG_TOY, Path(tmp) / "out", n_ctrl_delta=6, n_ctrl_embed=2, embed_alt=True)
    files = sorted(p.name for p in (Path(tmp) / "out").iterdir())
TOY_SECONDS = time.perf_counter() - t0
display(TOY_RES["delta"][["입력", "부분집합", "n", "delta", "delta_rel", "delta_bar"]])
display(TOY_RES["judge"]["hierarchy"]["table"])
display(TOY_RES["embed_table"][["입력", "기록", "쌍곡 이점", "왜곡 비", "수렴 쌍곡/유클리드", "시도 수"]])
print(f"합성 장난감 판정(의미 없음): {({k: v['advantage'] for k, v in TOY_RES['judge']['flow_advantage'].items()})} | 위계: {TOY_RES['judge']['hierarchy']['hierarchy']} | {TOY_SECONDS:.1f} s")
record("3절 배선: δ 표 6행, 대조군 δ̄ 4가지, 임베딩 7개, 판정, 저장 파일",
       len(TOY_RES["delta"]) == 6 and len(TOY_RES["ctrl_db"]) == 4 and len(TOY_RES["embed"]) == 7 and set(TOY_RES["judge"]["flow_advantage"]) == {"main", "alt"}
       and "delta_table.csv" in files and "embedding_table.csv" in files and any(f.startswith("worst20_") for f in files) and any(f.startswith("coords_") for f in files)
       and bool(TOY_RES["delta"]["delta_diff"].max() < 1e-9), n_files=len(files))

,입력,부분집합,n,delta,delta_rel,delta_bar
0,d_time_toy,전체,25,14.423741,0.291033,0.343213
1,d_time_toy,섬 제외,23,13.749538,0.475084,0.346540
2,d_flow_main,전체,25,2.725531,0.707575,0.499805
3,d_flow_main,섬 제외,23,2.725531,0.839262,0.507017
4,d_flow_alt,전체,25,2.509767,0.729983,0.488673
5,d_flow_alt,섬 제외,23,2.509767,0.781411,0.498017


,정규화,부분집합,d_flow,대조군 최솟값,대조군 중앙값,대조군 최댓값,d_flow 이하인 대조군 수,여유(최솟값 - d_flow),통과
0,main,전체,0.499805,0.491507,0.493240,0.496623,6,-0.008298,False
1,main,섬 제외,0.507017,0.501946,0.504093,0.504983,6,-0.005071,False
2,alt,전체,0.488673,0.481372,0.482328,0.484396,6,-0.007301,False
3,alt,섬 제외,0.498017,0.493293,0.494665,0.494978,6,-0.004724,False


,입력,기록,쌍곡 이점,왜곡 비,수렴 쌍곡/유클리드,시도 수
0,d_time_toy,쌍곡 이점 없음,쌍곡 이점 없음,0.933113,"5/5, 5/5",1
1,d_flow_main,쌍곡 이점 있음,쌍곡 이점 있음,0.663239,"5/5, 5/5",1
2,d_flow_alt,쌍곡 이점 있음,쌍곡 이점 있음,0.656743,"5/5, 5/5",1
3,ctrl_main_0,쌍곡 이점 있음,쌍곡 이점 있음,0.645349,"5/5, 5/5",1
4,ctrl_main_1,쌍곡 이점 있음,쌍곡 이점 있음,0.669552,"5/5, 5/5",1
5,ctrl_alt_0,쌍곡 이점 있음,쌍곡 이점 있음,0.661285,"5/5, 5/5",1
6,ctrl_alt_1,쌍곡 이점 있음,쌍곡 이점 있음,0.658986,"5/5, 5/5",1


합성 장난감 판정(의미 없음): {'main': '쌍곡 이점 없음', 'alt': '쌍곡 이점 있음'} | 위계: False | 14.1 s
통과  3절 배선: δ 표 6행, 대조군 δ̄ 4가지, 임베딩 7개, 판정, 저장 파일  {'n_files': 39}


In [10]:
# 1절 요약
assert all(v["ok"] for v in CHECKS.values())
print(f"합성 검산 {len(CHECKS)}개 모두 통과 (여기까지 {time.time() - T_START:.0f} s)")

합성 검산 43개 모두 통과 (여기까지 87 s)


## 2절. 입력 준비 (거리 행렬만. 임베딩·δ 계산 아님)

`d_time`(S2의 `d_sym`) 250개 단위와 249개 단위(창원 의창구·성산구를 합침, S3의 249개 대응), 관측 OD(DT_1PA2021, 249개 단위)와 `d_flow`(주·대안, 폐포), 중력 대조군 20개(주·대안), 섬 연결 지역을 만든다. 이 절은 δ·δ̄·임베딩을 계산하지 않는다. 산출물은 `C:\dev\kfm-data\spike\s4\`에 두고, 이미 있으면 내용이 같은지만 확인한다.

In [11]:
# 2-0. 입력 자료 MD5 점검 (manifest와 S3 고정 때 값과 같아야 한다)
manifest = pd.read_csv(RAW / "manifest.csv", encoding="utf-8-sig", dtype=str).set_index("path")
_od_rows = [p for p in manifest.index if "DT_1PA2021" in p]
assert len(_od_rows) == 1, _od_rows
OD_REL = _od_rows[0]  # manifest.csv에서 찾은 DT_1PA2021 원본 경로
print("OD 원본(manifest):", OD_REL)
S3 = json.loads((NB_DIR / "s3_freeze.json").read_text(encoding="utf-8"))
IN_FILES = {
    "travel_time.parquet": DATA / "travel_time.parquet",
    "regions.parquet": DATA / "regions.parquet",
    "island_links.csv": NB_DIR / "island_links.csv",
    "g_adj_edges.csv": NB_DIR / "g_adj_edges.csv",
    "g_adj249_edges.csv": NB_DIR / "g_adj249_edges.csv",
    "od_DT_1PA2021_csv": RAW / OD_REL,
    "s3_freeze.json": NB_DIR / "s3_freeze.json",
}
MD5 = {k: md5_file(p) for k, p in IN_FILES.items()}
assert MD5["od_DT_1PA2021_csv"] == manifest.loc[OD_REL, "md5"], "OD 원본이 manifest와 다르다"
for k in ("travel_time.parquet", "regions.parquet", "island_links.csv", "od_DT_1PA2021_csv"):
    assert MD5[k] == S3["inputs_md5"][k], f"{k}: S3 고정 때와 다르다"
for k in ("g_adj_edges.csv", "g_adj249_edges.csv"):
    assert MD5[k] == S3["outputs_md5"][k], f"{k}: S3 고정 때와 다르다"
print("입력 MD5 일치(manifest, S3 고정):", {k: v[:8] for k, v in MD5.items()})

OD 원본(manifest): kosis_DT_1PA2021_20261018/f_89098_101_DT_1PA2021_F_2020.csv
입력 MD5 일치(manifest, S3 고정): {'travel_time.parquet': '6a9e34cb', 'regions.parquet': 'a3332dbf', 'island_links.csv': 'e8db8aba', 'g_adj_edges.csv': 'd7ca076e', 'g_adj249_edges.csv': '8951efa9', 'od_DT_1PA2021_csv': '569f9db7', 's3_freeze.json': '606589e7'}


### 2-1. `d_time` 250개 단위

S2 `travel_time.parquet`의 `d_sym`(분)이 `d_time`이다. 방향 통행시간 `t_min`(i → j)은 중력 대조군의 β 적합에만 쓴다. 비대칭은 `t_min`에 그대로 두고 `d_sym`은 따로 둔다.

In [12]:
regions = pd.read_parquet(DATA / "regions.parquet", columns=["unit_code", "name", "sido_code", "pop_2025_12", "commute_code_2020", "unit249_code"])
CODES = regions["unit_code"].astype(str).to_numpy()
NAMES250 = regions["name"].to_numpy()
POP = regions["pop_2025_12"].to_numpy(float)
assert len(CODES) == 250 and list(CODES) == sorted(CODES)
tt = pd.read_parquet(DATA / "travel_time.parquet")
assert len(tt) == 250 * 250
T_DIR_250 = tt.pivot(index="from_id", columns="to_id", values="t_min").loc[CODES, CODES].to_numpy(float)
D_TIME_250 = tt.pivot(index="from_id", columns="to_id", values="d_sym").loc[CODES, CODES].to_numpy(float)
off250 = ~np.eye(250, dtype=bool)
assert np.abs(D_TIME_250 - D_TIME_250.T).max() < 1e-12 and np.allclose(D_TIME_250, (T_DIR_250 + T_DIR_250.T) / 2, atol=1e-9)
assert (np.diag(D_TIME_250) == 0).all() and (D_TIME_250[off250] > 0).all() and np.isfinite(D_TIME_250).all() and np.isfinite(T_DIR_250).all()
D_TIME_250 = (D_TIME_250 + D_TIME_250.T) / 2  # 비트 단위로 대칭
asym_idx = np.abs(T_DIR_250 - T_DIR_250.T)[off250] / (T_DIR_250 + T_DIR_250.T)[off250]
v_t, v_d = el.triangle_violation(T_DIR_250), el.triangle_violation(D_TIME_250)
assert v_t < 1e-6 and v_d < 1e-6
print(f"d_time 250: 최소 {D_TIME_250[off250].min():.2f} 분, 중앙값 {np.median(D_TIME_250[off250]):.1f}, 최대(직경) {D_TIME_250.max():.1f}; 방향 t_min 비대칭 지수 중앙값 {np.median(asym_idx):.4f}, 최대 {asym_idx.max():.3f}")
print(f"삼각 부등식 위반: 방향 {v_t:.1e}, d_sym {v_d:.1e} (분)")

d_time 250: 최소 2.69 분, 중앙값 155.9, 최대(직경) 621.4; 방향 t_min 비대칭 지수 중앙값 0.0029, 최대 0.321
삼각 부등식 위반: 방향 0.0e+00, d_sym 0.0e+00 (분)


### 2-2. `d_time` 249개 단위 (창원 의창구 48121 + 성산구 48123)

합친 지역의 거리는 하위 지역 쌍의 인구 가중 평균(주민등록 2025-12 인구, S3와 같은 가중)이다. 방향 통행시간도 같은 방식으로 만든다. 합치지 않은 단위는 250개 단위 값과 같고, S3 `g_adj249_edges.csv`의 엣지 `d_sym`과도 맞춘다.

In [13]:
U249 = regions["unit249_code"].astype(str).to_numpy()
CODES249 = sorted(set(U249))
assert len(CODES249) == 249
IDX249 = {c: i for i, c in enumerate(CODES249)}
GRP = np.array([IDX249[c] for c in U249])  # 250개 단위 -> 249개 단위
assert U249.tolist() == [("48121" if c == "48123" else c) for c in CODES.tolist()]
MERGED = np.bincount(GRP, minlength=249) > 1
assert MERGED.sum() == 1 and CODES249[int(np.flatnonzero(MERGED)[0])] == "48121"
POP249 = np.bincount(GRP, weights=POP, minlength=249)
D_TIME_249 = el.merge_weighted(D_TIME_250, GRP, POP)
T_DIR_249 = el.merge_weighted(T_DIR_250, GRP, POP)
rep_ = np.zeros(249, dtype=np.int64)
rep_[GRP] = np.arange(250)
single_ = np.flatnonzero(~MERGED)
assert np.abs(D_TIME_249[np.ix_(single_, single_)] - D_TIME_250[np.ix_(rep_[single_], rep_[single_])]).max() < 1e-9
assert np.abs(T_DIR_249[np.ix_(single_, single_)] - T_DIR_250[np.ix_(rep_[single_], rep_[single_])]).max() < 1e-9
NAMES249 = [str(NAMES250[rep_[i]]) for i in range(249)]
NAMES249[int(np.flatnonzero(MERGED)[0])] = "경상남도 창원시 의창구+성산구(합침)"
D_TIME_249 = (D_TIME_249 + D_TIME_249.T) / 2
off249 = ~np.eye(249, dtype=bool)
assert (D_TIME_249[off249] > 0).all() and np.allclose(D_TIME_249, (T_DIR_249 + T_DIR_249.T) / 2, atol=1e-9)

E249 = pd.read_csv(NB_DIR / "g_adj249_edges.csv", encoding="utf-8-sig", dtype={"u_code": str, "v_code": str}, usecols=["u_idx", "v_idx", "u_code", "v_code", "kind", "d_sym", "T_uv", "T_vu"])
assert (np.array(CODES249)[E249["u_idx"].to_numpy()] == E249["u_code"].to_numpy()).all() and (np.array(CODES249)[E249["v_idx"].to_numpy()] == E249["v_code"].to_numpy()).all()
assert np.abs(E249["d_sym"].to_numpy() - D_TIME_249[E249["u_idx"], E249["v_idx"]]).max() < 1e-9, "S3 249개 엣지의 d_sym과 다르다"
v249 = el.triangle_violation(D_TIME_249)
assert v249 < 1e-6
mi = int(np.flatnonzero(MERGED)[0])
print(f"d_time 249: 합친 단위 {CODES249[mi]} (인구 {POP249[mi]:,.0f}), 하위 지역 사이 d_sym {D_TIME_250[list(CODES).index('48121'), list(CODES).index('48123')]:.2f} 분")
print(f"S3 g_adj249 엣지 {len(E249)}개의 d_sym과 최대 차이 {np.abs(E249['d_sym'].to_numpy() - D_TIME_249[E249['u_idx'], E249['v_idx']]).max():.1e}; 삼각 부등식 위반 {v249:.1e}")

d_time 249: 합친 단위 48121 (인구 454,045), 하위 지역 사이 d_sym 18.43 분
S3 g_adj249 엣지 660개의 d_sym과 최대 차이 2.8e-14; 삼각 부등식 위반 0.0e+00


### 2-3. 섬 연결 지역 (ADR 0010 5항)

`G_adj`(경계 공유 + 연륙·연도교 + 섬 링크)에서 섬 링크 엣지를 모두 뺐을 때 가장 큰 연결 성분(본토)에 들지 않는 지역이다. 연륙·연도교 엣지는 빼지 않는다. 250개 단위와 249개 단위를 각각 구한다.

In [14]:
E250 = pd.read_csv(NB_DIR / "g_adj_edges.csv", encoding="utf-8-sig", usecols=["u_idx", "v_idx", "kind"])
ISL250 = el.island_connected_mask(250, E250["u_idx"].to_numpy(), E250["v_idx"].to_numpy(), E250["kind"].to_numpy())
ISL249 = el.island_connected_mask(249, E249["u_idx"].to_numpy(), E249["v_idx"].to_numpy(), E249["kind"].to_numpy())
assert np.array_equal(ISL249, np.bincount(GRP, weights=ISL250.astype(float), minlength=249) > 0), "250개와 249개 단위의 섬 연결 지역이 서로 다르다"
print("G_adj 엣지 종류(250):", E250["kind"].value_counts().to_dict(), "| (249):", E249["kind"].value_counts().to_dict())
print("섬 연결 지역(250개 단위):", [(CODES[i], str(NAMES250[i])) for i in np.flatnonzero(ISL250)])
print("섬 연결 지역(249개 단위):", [(CODES249[i], NAMES249[i]) for i in np.flatnonzero(ISL249)])

G_adj 엣지 종류(250): {'boundary': 638, 'bridge': 22, 'island': 3} | (249): {'boundary': 635, 'bridge': 22, 'island': 3}
섬 연결 지역(250개 단위): [('47940', '경상북도 울릉군'), ('50110', '제주특별자치도 제주시'), ('50130', '제주특별자치도 서귀포시')]
섬 연결 지역(249개 단위): [('47940', '경상북도 울릉군'), ('50110', '제주특별자치도 제주시'), ('50130', '제주특별자치도 서귀포시')]


### 2-4. 관측 OD (DT_1PA2021, 2020 통근·통학, 249개 단위)

원본(manifest MD5 확인)을 250개 단위로 읽어 S3의 249개 대응으로 합친다. 합친 지역 안의 통행은 존 내 통행으로 보아 뺀다. 표본조사(20%) 확대 추정치다. 이 절은 OD 요약과 쌍방향 통행 그래프의 연결 여부만 본다.

In [15]:
rows_ = [r for r in csv.reader(io.StringIO((RAW / OD_REL).read_bytes().decode("cp949"))) if len(r) == 6]
od_ = pd.DataFrame(rows_[1:], columns=["o_code", "o_name", "d_code", "d_name", "year", "value"])
for c_ in ("o_code", "d_code"):
    od_[c_] = od_[c_].str.lstrip("'")
od_["value"] = od_["value"].astype(float)
leaf2idx = {c: i for i, c in enumerate(regions["commute_code_2020"].astype(str))}
assert len(leaf2idx) == 250
od_leaf = od_[od_["o_code"].isin(leaf2idx) & od_["d_code"].isin(leaf2idx)]
OD_TOTAL = float(od_.loc[(od_["o_code"] == "00") & (od_["d_code"] == "00"), "value"].iloc[0])
assert OD_TOTAL == 28012287 and od_leaf["value"].sum() == OD_TOTAL, "전국 합계(28,012,287)와 다르다"
T250 = np.zeros((250, 250))
np.add.at(T250, (od_leaf["o_code"].map(leaf2idx).to_numpy(), od_leaf["d_code"].map(leaf2idx).to_numpy()), od_leaf["value"].to_numpy())
assert T250.sum() == OD_TOTAL
T249 = el.merge_sum(T250, GRP)                                  # 존 내 통행(대각) 제외
INTRA249 = np.diag(el.merge_sum(T250, GRP, zero_diag=False))     # 존 내 통행(합친 지역 안의 통행 포함)
assert abs(T249.sum() + INTRA249.sum() - OD_TOTAL) < 1e-6
assert np.abs(E249["T_uv"].to_numpy() - T249[E249["u_idx"], E249["v_idx"]]).max() == 0 and np.abs(E249["T_vu"].to_numpy() - T249[E249["v_idx"], E249["u_idx"]]).max() == 0, "S3 249개 엣지의 흐름과 다르다"


def od_stats(T):
    n = len(T)
    T0, O, Dm, V = el.od_margins(T)
    off = ~np.eye(n, dtype=bool)
    pos = T0 > 0
    nc, lab = el.flow_components(T0)
    return {"n": n, "inter_zone_sum": float(T0.sum()), "offdiag_cells": int(off.sum()), "zero_cells": int((~pos & off).sum()),
            "zero_cell_frac": float((~pos & off).sum() / off.sum()), "bidirectional_pairs": int((pos & pos.T).sum() // 2),
            "one_direction_only_pairs": int((pos ^ pos.T).sum() // 2), "components": int(nc), "component_sizes": np.bincount(lab).tolist(),
            "O_min": float(O.min()), "D_min": float(Dm.min()), "zero_O": int((O == 0).sum()), "zero_D": int((Dm == 0).sum())}


OD249 = od_stats(T249)
OD249.update({"total_incl_intra": OD_TOTAL, "intra_zone_sum": float(INTRA249.sum()), "value_min_positive": float(T249[T249 > 0].min()), "value_max": float(T249.max())})
print("관측 OD(249개 단위):", OD249)
print(f"  전국 합계(존 내 포함) {OD_TOTAL:,.0f}명, 존 간 합 {OD249['inter_zone_sum']:,.0f}명, 존 간 셀 {OD249['offdiag_cells']:,}개 중 0인 셀 {OD249['zero_cells']:,}개 ({100 * OD249['zero_cell_frac']:.2f}%)")
print(f"  양방향 양수 쌍 {OD249['bidirectional_pairs']:,}개(전체 쌍 {249 * 248 // 2:,}개), 한 방향만 양수인 쌍 {OD249['one_direction_only_pairs']:,}개, 쌍방향 그래프 연결 성분 {OD249['components']}개 {OD249['component_sizes']}")

관측 OD(249개 단위):

 {'n': 249, 'inter_zone_sum': 9520502.0, 'offdiag_cells': 61752, 'zero_cells': 44497, 'zero_cell_frac': 0.7205758517942739, 'bidirectional_pairs': 6816, 'one_direction_only_pairs': 3623, 'components': 2, 'component_sizes': [247, 2], 'O_min': 36.0, 'D_min': 82.0, 'zero_O': 0, 'zero_D': 0, 'total_incl_intra': 28012287.0, 'intra_zone_sum': 18491785.0, 'value_min_positive': 1.0, 'value_max': 40783.0}
  전국 합계(존 내 포함) 28,012,287명, 존 간 합 9,520,502명, 존 간 셀 61,752개 중 0인 셀 44,497개 (72.06%)
  양방향 양수 쌍 6,816개(전체 쌍 30,876개), 한 방향만 양수인 쌍 3,623개, 쌍방향 그래프 연결 성분 2개 [247, 2]


### 2-5. OD에서 빠진 지역과 관측 `d_flow` (주 ℓ_sym, 대안 ℓ′, 폐포)

ADR 0010 2항(2026-10-19 보강 포함): OD 자료에서 빠진 지역은 미리 빼고 계산하며 목록을 기록한다. "통근·통학 OD처럼 바다를 건너는 일상 통행이 거의 없는 자료에서는 섬 연결 지역 중 본토와 양방향 통행이 없는 지역도 도서지역 제외로 미리 뺀다(제주시·서귀포시)." 그 밖의 이유로 연결 성분이 둘 이상이면 중단한다. 위 쌍방향 그래프는 연결 성분이 둘이다. 작은 성분의 지역과 그 통행을 아래에 보인다. `flow_exclusion`은 (1) 존 간 통행이 전혀 없는 지역, (2) 본토 성분 밖에 있으면서 섬 연결 지역인 지역을 빼고, 그 밖에 끊어진 지역이 있으면 중단한다. 이 절도 δ·δ̄는 계산하지 않는다.

In [16]:
nc0, lab0 = el.flow_components(T249)
main0 = int(np.argmax(np.bincount(lab0)))
small = np.flatnonzero(lab0 != main0)
print(f"쌍방향 그래프(249개 단위) 연결 성분 {nc0}개, 크기 {np.bincount(lab0).tolist()}; 본토 성분 밖 지역:")
for i in small:
    out_ = [(NAMES249[j], float(T249[i, j])) for j in np.flatnonzero(T249[i] > 0)]
    in_ = [(NAMES249[j], float(T249[j, i])) for j in np.flatnonzero(T249[:, i] > 0)]
    print(f"  {CODES249[i]} {NAMES249[i]}: O = {T249[i].sum():,.0f}, D = {T249[:, i].sum():,.0f}, 섬 연결 지역 = {bool(ISL249[i])}\n    나가는 통행 {out_}\n    들어오는 통행 {in_}")
UNLINKED = el.od_unlinked_units(T249)
print("존 간 통행이 전혀 없는 지역:", [(CODES249[i], NAMES249[i]) for i in UNLINKED], "(없으면 빈 목록)")
FX = el.flow_exclusion(T249, ISL249)
KEEP247 = FX["keep"]
IDX247 = np.flatnonzero(KEEP247)
assert len(IDX247) == 247 and sorted(FX["island_dropped"].tolist()) == sorted(small.tolist()) and len(FX["unlinked"]) == 0
CODES247 = [CODES249[i] for i in IDX247]
NAMES247 = [NAMES249[i] for i in IDX247]
EXCLUDED = [{"code": CODES249[i], "name": NAMES249[i], "reason": "도서지역 제외(ADR 0010 2항 보강): 쌍방향 통행 그래프에서 본토 성분 밖인 섬 연결 지역"} for i in FX["island_dropped"]]
print("d_flow에서 뺀 지역:", EXCLUDED)
T247 = T249[np.ix_(IDX247, IDX247)]
D_TIME_247 = el.subset_matrix(D_TIME_249, IDX247)
T_DIR_247 = el.subset_matrix(T_DIR_249, IDX247)
ISL247 = ISL249[IDX247]
OD247 = od_stats(T247)
print("관측 OD(247개 단위):", OD247)
print("247개 단위의 섬 연결 지역:", [(CODES247[i], NAMES247[i]) for i in np.flatnonzero(ISL247)])
assert OD247["components"] == 1 and OD247["zero_O"] == 0 and OD247["zero_D"] == 0

FDOBS = el.flow_distance(T247, kind="observed", role="observed")  # 관측 모드. 연결 성분이 둘 이상이면 여기서 멈춘다
D_FLOW_MAIN, D_FLOW_ALT = FDOBS["d_main"], FDOBS["d_alt"]
off247 = ~np.eye(247, dtype=bool)
for nm_, M_, raw_ in (("주 ℓ_sym", D_FLOW_MAIN, FDOBS["raw_main"]), ("대안 ℓ′", D_FLOW_ALT, FDOBS["raw_alt"])):
    assert np.isfinite(M_).all() and (M_[off247] > 0).all() and np.allclose(M_, M_.T) and np.diag(M_).max() == 0
    short_ = int(((M_ < raw_ - 1e-12) & FDOBS["edge"]).sum() // 2)
    print(f"d_flow {nm_}: 엣지 {int(FDOBS['edge'].sum() // 2):,}개, 폐포 후 최소 {M_[off247].min():.4f}, 중앙값 {np.median(M_[off247]):.3f}, 최대(직경) {M_.max():.3f}; "
          f"폐포로 줄어든 엣지 {short_}개, 삼각 부등식 위반 {el.triangle_violation(M_):.1e}, 연결 성분 1개")

쌍방향 그래프(249개 단위) 연결 성분 2개, 크기 [247, 2]; 본토 성분 밖 지역:
  50110 제주특별자치도 제주시: O = 11,197, D = 7,366, 섬 연결 지역 = True
    나가는 통행 [('제주특별자치도 서귀포시', 11197.0)]
    들어오는 통행 [('인천광역시 남동구', 9.0), ('제주특별자치도 서귀포시', 7357.0)]
  50130 제주특별자치도 서귀포시: O = 7,357, D = 11,197, 섬 연결 지역 = True
    나가는 통행 [('제주특별자치도 제주시', 7357.0)]
    들어오는 통행 [('제주특별자치도 제주시', 11197.0)]
존 간 통행이 전혀 없는 지역: [] (없으면 빈 목록)
d_flow에서 뺀 지역: [{'code': '50110', 'name': '제주특별자치도 제주시', 'reason': '도서지역 제외(ADR 0010 2항 보강): 쌍방향 통행 그래프에서 본토 성분 밖인 섬 연결 지역'}, {'code': '50130', 'name': '제주특별자치도 서귀포시', 'reason': '도서지역 제외(ADR 0010 2항 보강): 쌍방향 통행 그래프에서 본토 성분 밖인 섬 연결 지역'}]
관측 OD(247개 단위): {'n': 247, 'inter_zone_sum': 9501939.0, 'offdiag_cells': 60762, 'zero_cells': 43510, 'zero_cell_frac': 0.716072545340838, 'bidirectional_pairs': 6815, 'one_direction_only_pairs': 3622, 'components': 1, 'component_sizes': [247], 'O_min': 36.0, 'D_min': 82.0, 'zero_O': 0, 'zero_D': 0}
247개 단위의 섬 연결 지역: [('47940', '경상북도 울릉군')]
d_flow 주 ℓ_sym: 엣지 6,815개, 폐포 후 최소 0.3178, 중

### 2-6. 중력 대조군 (ADR 0010 6항)

247개 단위의 관측 OD로 이중 제약 지수 중력모형을 적합한다: 발생·도착 합은 존 내 통행을 뺀 O_i, D_j, β는 존 내 통행을 뺀 관측 T로 가중한 방향 `d_time`(i → j)의 평균에 맞추고, 균형은 Furness. 존 간 합 × 0.2(반올림)를 총량으로 존 간 셀을 다항 재표본하고, ×5 확대한 뒤 관측의 0 패턴을 적용한다. 대조군 r은 시드 r이고, 연결 성분이 둘 이상인 시드는 건너뛰어 연결된 20개를 모은다(주·대안 두 정규화가 같은 엣지 집합이므로 연결 여부가 같다).

In [17]:
FIT = el.fit_gravity(T247, T_DIR_247)
print(f"중력모형 적합: β = {FIT['beta']:.8f} /분, 평균 방향 통행시간 관측 {FIT['mean_time_obs']:.6f} 분 = 모형 {FIT['mean_time_fit']:.6f} 분, Furness {FIT['furness_iter']}회, "
      f"행합 최대 상대 오차 {FIT['row_sum_max_rel_err']:.1e}, 열합 {FIT['col_sum_max_rel_err']:.1e}")
CTRL, SKIPPED = el.make_controls(FIT["T_exp"], T247, n_wanted=20, cfg=CFG)
N_EFF = CTRL[0]["n_eff"]
print(f"재표본 총량 N_eff = round(0.2 x {OD247['inter_zone_sum']:,.0f}) = {N_EFF:,} (대조군 합 = {5 * N_EFF:,} 이하)")
print("연결된 대조군 시드:", [c["seed"] for c in CTRL])
print("건너뛴 시드:", [s["seed"] for s in SKIPPED], "| 사유:", sorted({s["reason"].split(". ")[0] for s in SKIPPED}))
obs_edges = int(FDOBS["edge"].sum() // 2)
pos_obs = T247 > 0
rows_c = []
for c in CTRL:
    Tc_ = c["T"]
    assert np.all(Tc_ % 5 == 0) and np.all(np.diag(Tc_) == 0) and np.all(Tc_[T247 == 0] == 0) and c["n_eff"] == N_EFF
    for key_ in ("d_main", "d_alt"):
        assert np.isfinite(c[key_]).all() and (c[key_][off247] > 0).all() and np.allclose(c[key_], c[key_].T)
    O_c, D_c = Tc_.sum(axis=1), Tc_.sum(axis=0)
    rows_c.append({"seed": c["seed"], "합": Tc_.sum(), "엣지": c["n_edges"], "관측 엣지 대비": c["n_edges"] / obs_edges,
                   "0 셀 비율": float(((Tc_ == 0) & off247).sum() / off247.sum()),
                   "O 상대 오차 중앙값": float(np.median(np.abs(O_c / FDOBS["O"] - 1))), "D 상대 오차 중앙값": float(np.median(np.abs(D_c / FDOBS["D"] - 1)))})
display(pd.DataFrame(rows_c).set_index("seed").round(4))
print(f"관측: 합 {OD247['inter_zone_sum']:,.0f}, 엣지 {obs_edges:,}, 0 셀 비율 {OD247['zero_cell_frac']:.4f}")

중력모형 적합: β = 0.10015664 /분, 평균 방향 통행시간 관측 21.451213 분 = 모형 21.451213 분, Furness 5261회, 행합 최대 상대 오차 1.0e-13, 열합 2.9e-15


재표본 총량 N_eff = round(0.2 x 9,501,939) = 1,900,388 (대조군 합 = 9,501,940 이하)
연결된 대조군 시드: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
건너뛴 시드: [] | 사유: []


,합,엣지,관측 엣지 대비,0 셀 비율,O 상대 오차 중앙값,D 상대 오차 중앙값
seed,,,,,,
0,9487170.0,5233,0.7679,0.7979,0.0112,0.0118
1,9487380.0,5187,0.7611,0.7978,0.0114,0.0110
2,9487080.0,5186,0.7610,0.7986,0.0122,0.0108
3,9487430.0,5197,0.7626,0.7983,0.0096,0.0114
4,9486790.0,5228,0.7671,0.7974,0.0123,0.0112
5,9487450.0,5225,0.7667,0.7980,0.0119,0.0107
6,9487240.0,5228,0.7671,0.7981,0.0105,0.0108
7,9487130.0,5217,0.7655,0.7983,0.0116,0.0101
8,9487565.0,5227,0.7670,0.7977,0.0147,0.0105


관측: 합 9,501,939, 엣지 6,815, 0 셀 비율 0.7161


### 2-7. 산출물 저장, 3절 입력 구성

행렬은 `float64` `.npy`로 `C:\dev\kfm-data\spike\s4\`에 둔다. 이미 있으면 내용이 같은지 확인한다(고정 뒤에는 바뀌면 안 된다). 대조군 20개는 쌓은 배열 하나로 저장한다.

In [18]:
OUT_MD5 = {}
OUT_MD5["d_time_250.npy"] = emit_npy(OUT / "d_time_250.npy", D_TIME_250)
OUT_MD5["t_dir_250.npy"] = emit_npy(OUT / "t_dir_250.npy", T_DIR_250)
OUT_MD5["d_time_249.npy"] = emit_npy(OUT / "d_time_249.npy", D_TIME_249)
OUT_MD5["t_dir_249.npy"] = emit_npy(OUT / "t_dir_249.npy", T_DIR_249)
OUT_MD5["d_time_247.npy"] = emit_npy(OUT / "d_time_247.npy", D_TIME_247)
OUT_MD5["t_dir_247.npy"] = emit_npy(OUT / "t_dir_247.npy", T_DIR_247)
OUT_MD5["od_249.npy"] = emit_npy(OUT / "od_249.npy", T249)
OUT_MD5["od_247.npy"] = emit_npy(OUT / "od_247.npy", T247)
OUT_MD5["d_flow_main_247.npy"] = emit_npy(OUT / "d_flow_main_247.npy", D_FLOW_MAIN)
OUT_MD5["d_flow_alt_247.npy"] = emit_npy(OUT / "d_flow_alt_247.npy", D_FLOW_ALT)
OUT_MD5["gravity_expected_247.npy"] = emit_npy(OUT / "gravity_expected_247.npy", FIT["T_exp"])
OUT_MD5["controls_T_247.npy"] = emit_npy(OUT / "controls_T_247.npy", np.stack([c["T"] for c in CTRL]))
OUT_MD5["controls_d_main_247.npy"] = emit_npy(OUT / "controls_d_main_247.npy", np.stack([c["d_main"] for c in CTRL]))
OUT_MD5["controls_d_alt_247.npy"] = emit_npy(OUT / "controls_d_alt_247.npy", np.stack([c["d_alt"] for c in CTRL]))
units250 = pd.DataFrame({"idx": np.arange(250), "unit_code": CODES, "name": NAMES250, "pop_2025_12": POP, "island_connected": ISL250, "idx249": GRP})
units249 = pd.DataFrame({"idx": np.arange(249), "unit_code": CODES249, "name": NAMES249, "pop_2025_12": POP249, "island_connected": ISL249,
                         "in_flow_set": KEEP247, "idx247": np.cumsum(KEEP247) - 1})
units249.loc[~KEEP247, "idx247"] = -1
units247 = pd.DataFrame({"idx": np.arange(247), "idx249": IDX247, "unit_code": CODES247, "name": NAMES247, "island_connected": ISL247})
OUT_MD5["units_250.csv"] = emit_csv(OUT / "units_250.csv", units250)
OUT_MD5["units_249.csv"] = emit_csv(OUT / "units_249.csv", units249)
OUT_MD5["units_247.csv"] = emit_csv(OUT / "units_247.csv", units247)
CONTROLS_META = {"seeds": [c["seed"] for c in CTRL], "skipped_seeds": [s["seed"] for s in SKIPPED], "skipped_reasons": {str(s["seed"]): s["reason"] for s in SKIPPED},
                 "n_eff": int(N_EFF), "edges": [c["n_edges"] for c in CTRL], "beta_per_min": FIT["beta"], "units": 247}
OUT_MD5["controls_meta.json"] = emit_json(OUT / "controls_meta.json", CONTROLS_META)

DATA_REAL = {
    "delta_inputs": {
        "d_time_250": {"D": D_TIME_250, "island": ISL250},
        "d_time_249": {"D": D_TIME_249, "island": ISL249},
        "d_time_247": {"D": D_TIME_247, "island": ISL247},
        "d_flow_main": {"D": D_FLOW_MAIN, "island": ISL247},
        "d_flow_alt": {"D": D_FLOW_ALT, "island": ISL247},
    },
    "flow_names": {"main": "d_flow_main", "alt": "d_flow_alt"},
    "flow_island": ISL247,
    "controls": {"main": [c["d_main"] for c in CTRL], "alt": [c["d_alt"] for c in CTRL]},
    "embed_inputs": {"d_time_250": D_TIME_250, "d_time_247": D_TIME_247, "d_flow_main": D_FLOW_MAIN, "d_flow_alt": D_FLOW_ALT},
}
LABELS = {250: [str(x) for x in NAMES250], 247: NAMES247}
assert len(DATA_REAL["controls"]["main"]) == 20 and all(m.shape == (247, 247) for m in DATA_REAL["controls"]["alt"])
# 임베딩 입력 검증만 한다(대칭, 대각 0, 서로 다른 쌍은 양의 유한값). 임베딩은 계산하지 않는다.
for name_, D_ in el.embedding_problems(DATA_REAL, 5, True)[0].items():
    pr_ = el.Problem(D_)
    assert pr_.n in (247, 250) and len(pr_.d) == pr_.n * (pr_.n - 1) // 2
print("3절 입력:", {k: (len(v) if isinstance(v, (dict, list)) else None) for k, v in DATA_REAL.items()}, "| 임베딩 입력 14개의 형식 검증 통과")

  검증: d_time_250.npy (기존 파일과 같다)
  검증: t_dir_250.npy (기존 파일과 같다)
  검증: d_time_249.npy (기존 파일과 같다)
  검증: t_dir_249.npy (기존 파일과 같다)
  검증: d_time_247.npy (기존 파일과 같다)
  검증: t_dir_247.npy (기존 파일과 같다)
  검증: od_249.npy (기존 파일과 같다)
  검증: od_247.npy (기존 파일과 같다)
  검증: d_flow_main_247.npy (기존 파일과 같다)
  검증: d_flow_alt_247.npy (기존 파일과 같다)
  검증: gravity_expected_247.npy (기존 파일과 같다)
  검증: controls_T_247.npy (기존 파일과 같다)
  검증: controls_d_main_247.npy (기존 파일과 같다)
  검증: controls_d_alt_247.npy (기존 파일과 같다)
  검증: units_250.csv (기존 파일과 같다)
  검증: units_249.csv (기존 파일과 같다)
  검증: units_247.csv (기존 파일과 같다)
  검증: controls_meta.json (기존 파일과 같다)
3절 입력: {'delta_inputs': 5, 'flow_names': 2, 'flow_island': None, 'controls': 2, 'embed_inputs': 4} | 임베딩 입력 14개의 형식 검증 통과


### 2-8. 사전 고정 기록 `s4_freeze.json`

입력 MD5, 설정값(m₁ = 0.5, m₂ = 0.9, 최적화 설정, 시드 규칙), 합성 검산 요약, β, 건너뛴 시드, 제외 지역, 섬 연결 지역을 담는다. 이 파일과 노트북, `embed_lib.py`를 실데이터 임베딩·δ·판정 전에 커밋한다. 이미 있으면 내용이 같은지 확인한다(2부 실행에서 다시 검증한다).

In [19]:
OUT_MD5["embed_lib.py"] = md5_file(NB_DIR / "embed_lib.py")
FREEZE = {
    "adr": "ADR 0010 2·5·6·7항(채택 2026-10-19 10:08 KST, 태그 0S-start), ADR 0013 4항. S4 실데이터 임베딩·δ·판정 전 사전 고정",
    "created": "2026-10-19",
    "session": "S4 1부 (0S-4 임베딩·δ·d_flow·중력 대조군 코드, 합성 검산, 입력 거리 행렬 준비)",
    "decisions_2026_10_19": {
        "a_d_flow_units": "d_flow는 247개 단위(제주시·서귀포시 제외). ADR 0010 2항 보강: 통근·통학 OD처럼 바다를 건너는 일상 통행이 거의 없는 자료에서는 섬 연결 지역 중 본토와 양방향 통행이 없는 지역도 도서지역 제외로 미리 뺀다",
        "b_retry": "재실행 규약(ADR 0013 4항 보강): 쌍곡 U(±0.1) 재시작 3개만 시드 +5, +10으로 최대 2회. 유클리드 결과와 줄인 초기값 2개는 그대로. 쌍곡은 모든 시도의 재시작을 합쳐 판정, 실패 판정은 시도마다 합친 쌍곡 최솟값. 실패·평탄 극한 구분은 수렴과 무관한 최소 stress",
        "c_inputs": "임베딩 입력 14개(d_time 250·247, d_flow 주·대안, 대조군 앞 5개의 주·대안). d_time 249개 단위는 δ만. MAP 제외",
    },
    "environment": {"python": sys.version.split()[0], "numpy": np.__version__, "scipy": scipy.__version__, "pandas": pd.__version__,
                    "torch": torch.__version__, "geoopt": geoopt.__version__, "numba": el.numba.__version__, "env": "C:/dev/kfm-spike (numba는 S4에서 더함)"},
    "config": {
        "embedding": {k: v for k, v in CFG.items()},
        "m1": M1, "m2": M2,
        "config_hash": el.config_hash(CFG),
        "seed_rules": SEED_RULES,
        "precision": "float64, CPU 1스레드/프로세스, PYTORCH_JIT=0(재시작 결과가 호출 순서에 의존하지 않게), geoopt PoincareBall(c=1), torch.set_default_dtype(float64)",
        "lbfgsb_unspecified_by_adr": {"gtol": CFG["fin_gtol"], "maxfun": CFG["fin_maxfun"], "reason": "gtol 1e-10: ftol이 실제 종료 기준이 되게(S1b), maxfun 50,000: 기본 15,000은 maxiter보다 작다"},
    },
    "rules": {
        "hyperbolic_advantage": "마무리 수렴 재시작 중 상대 stress 최소의 평균 상대 왜곡 비(쌍곡/유클리드) <= 0.8. d_flow는 대조군 5개 비의 최솟값보다 작아야 한다. 대조군 중 하나라도 판정 재시작이 없으면 보류",
        "flat_limit": "쌍곡 최소 stress가 유클리드 최소 stress의 1~(1+1e-3)배: '쌍곡 이점 없음(평탄 극한)'(수렴 불필요). 그보다 크면 최적화 실패 -> 쌍곡 U 재시작 3개만 시드 +5, +10으로 최대 2회 재실행(유클리드·줄인 초기값은 그대로), 쌍곡은 모든 시도를 합쳐 판정, 실패 판정은 시도마다 합친 쌍곡 최솟값 -> 두 번 재실행해도 실패면 '최적화 실패'·보류",
        "hierarchy": "delta_bar(d_flow) < 대조군 20개 모두. 주·대안 정규화 각각, 전체·섬 제외 각각(4개 조건)",
        "delta": "서로 다른 네 점, delta = max (S1-S2)/2 (기준점별 (max,min) 행렬곱, 정확값), delta_rel = 2 delta / diam, delta_bar = mean (S1-S2)/(S1-S3) (세 합이 같으면 0, 상대 오차 1e-12 이내)",
        "subsets": "부분집합은 전체 행렬의 부분행렬(폐포 다시 안 함). 섬 제외 = G_adj에서 섬 링크 엣지를 뺐을 때 본토 밖 지역",
    },
    "units": {"d_time_250": 250, "d_time_249": 249, "d_time_247": 247, "d_flow": 247, "embedded_inputs": sorted(DATA_REAL["embed_inputs"]) + [f"ctrl_{nm}_{r}" for nm in el.NORMS for r in range(5)]},
    "island_connected": {
        "250": [{"code": CODES[i], "name": str(NAMES250[i])} for i in np.flatnonzero(ISL250)],
        "249": [{"code": CODES249[i], "name": NAMES249[i]} for i in np.flatnonzero(ISL249)],
        "247": [{"code": CODES247[i], "name": NAMES247[i]} for i in np.flatnonzero(ISL247)],
    },
    "observed_od": {"source": "KOSIS DT_1PA2021 인구총조사 2020 통근·통학(12세 이상, 20% 표본 확대 추정치), 249개 단위(창원 의창+성산 합침), 존 내 통행 제외",
                    "units_249": OD249, "units_247": OD247},
    "flow_exclusion": {"unlinked_regions": [{"code": CODES249[i], "name": NAMES249[i]} for i in FX["unlinked"]], "island_dropped": EXCLUDED,
                       "rule": "ADR 0010 2항(2026-10-19 보강): 빠진 지역은 미리 빼고 기록. 통근·통학 OD처럼 바다를 건너는 일상 통행이 거의 없는 자료에서는 섬 연결 지역 중 본토와 양방향 통행이 없는 지역도 도서지역 제외로 미리 뺀다(제주시·서귀포시). 제외 허용 = 존 간 통행 없음, 또는 쌍방향 그래프 본토 성분 밖인 섬 연결 지역. 그 밖의 단절은 중단"},
    "d_flow": {"edges": int(FDOBS["edge"].sum() // 2), "components": 1, "units": 247},
    "gravity": {"beta_per_min": FIT["beta"], "mean_time_obs_min": FIT["mean_time_obs"], "mean_time_fit_min": FIT["mean_time_fit"], "furness_iter": FIT["furness_iter"],
                "n_eff": int(N_EFF), "resample_frac": CFG["resample_frac"], "scale": CFG["resample_scale"],
                "control_seeds": [c["seed"] for c in CTRL], "skipped_seeds": [s["seed"] for s in SKIPPED], "control_edges": [c["n_edges"] for c in CTRL]},
    "inputs_md5": MD5,
    "outputs_md5": OUT_MD5,
    "synthetic_checks": CHECKS,
    "run_real_when_frozen": False,
}
emit_json(NB_DIR / "s4_freeze.json", FREEZE)
print("고정 값 요약: m1, m2 =", M1, M2, "| β =", round(FIT["beta"], 8), "| 건너뛴 시드", [s["seed"] for s in SKIPPED], "| 제외 지역", [e["name"] for e in EXCLUDED],
      "| 합성 검산", len(CHECKS), "개 통과")

  검증: s4_freeze.json (기존 파일과 같다)
고정 값 요약: m1, m2 = 0.5 0.9 | β = 0.10015664 | 건너뛴 시드 [] | 제외 지역 ['제주특별자치도 제주시', '제주특별자치도 서귀포시'] | 합성 검산 43 개 통과


### 2-9. 3절 실행 시간 추정

tree 실행 시간(1-1)에서 환산한다. 반복 수를 모르므로 세 가지로 본다. (A) tree의 재시작 시간을 쌍 수(247개 단위 30,381쌍 / tree 465쌍)에 비례시킨 값, (B) 합성 점 250개에서 잰 한 반복의 시간 × tree와 같은 반복 수, (C) 반복 상한(Adam 50,000 + 마무리 20,000회). 임베딩 입력 14개 × (쌍곡 5 + 유클리드 5)회이고, 재실행이 필요하면 쌍곡 U 재시작 3개씩 최대 2회(입력당 쌍곡 최대 11회, 유클리드는 그대로)가 더해진다. δ·δ̄ 계산(입력 5개 × 2 부분집합 + 대조군 80회)은 분 단위다.

In [20]:
BENCH = el.bench_iteration_cost(n=250, n_adam=200, n_eval=100)  # 합성 점(실데이터 아님)에서 한 반복의 시간
pairs_ = lambda n: n * (n - 1) // 2  # noqa: E731
PROBS, _ = el.embedding_problems(DATA_REAL, 5, True)
n_prob = len(PROBS)
scale = pairs_(247) / TREE_STATS["n_pairs"]
ph, pe = BENCH["poincare"], BENCH["euclid"]
est = {
    "A 쌍 수 비례(tree 시간 x %.0f)" % scale: (TREE_STATS["hyp_sec_mean"] * scale, TREE_STATS["euc_sec_mean"] * scale),
    "B n=250 한 반복 시간 x tree 반복 수": (TREE_STATS["hyp_adam_iter_mean"] * ph["sec_per_adam_iter"] + TREE_STATS["hyp_fin_iter_mean"] * ph["sec_per_eval"],
                                       TREE_STATS["euc_adam_iter_mean"] * pe["sec_per_adam_iter"] + TREE_STATS["euc_fin_iter_mean"] * pe["sec_per_eval"]),
    "C 반복 상한(Adam 5만 + 마무리 2만)": (50_000 * ph["sec_per_adam_iter"] + 20_000 * ph["sec_per_eval"], 50_000 * pe["sec_per_adam_iter"] + 20_000 * pe["sec_per_eval"]),
}
rows_e = []
for name_, (h_s, e_s) in est.items():
    serial = n_prob * (5 * h_s + 5 * e_s)
    rows_e.append({"추정": name_, "쌍곡 재시작(초)": round(h_s), "유클리드 재시작(초)": round(e_s), "직렬 합(시간)": round(serial / 3600, 2),
                   f"병렬 {N_WORKERS}개(시간, 이상적)": round(serial / 3600 / N_WORKERS, 2), "재실행 최악: 쌍곡 11회(병렬, 시간)": round(n_prob * (11 * h_s + 5 * e_s) / 3600 / N_WORKERS, 2)})
EST = pd.DataFrame(rows_e).set_index("추정")
print(f"tree 10회 최적화(시드 0-4, 두 공간) {TREE_STATS['seconds_total']:.1f} s: 쌍곡 재시작 평균 {TREE_STATS['hyp_sec_mean']:.1f} s (Adam {TREE_STATS['hyp_adam_iter_mean']:.0f}회 + 마무리 {TREE_STATS['hyp_fin_iter_mean']:.0f}회), "
      f"유클리드 평균 {TREE_STATS['euc_sec_mean']:.1f} s (Adam {TREE_STATS['euc_adam_iter_mean']:.0f}회)")
print(f"합성 점 n=250(쌍 {BENCH['n_pairs']:,}): 쌍곡 Adam 한 단계 {1e3 * ph['sec_per_adam_iter']:.2f} ms, 함수·기울기 {1e3 * ph['sec_per_eval']:.2f} ms; "
      f"유클리드 {1e3 * pe['sec_per_adam_iter']:.2f} / {1e3 * pe['sec_per_eval']:.2f} ms")
print(f"임베딩 입력 {n_prob}개 x (쌍곡 5 + 유클리드 5) 재시작")
display(EST)

tree 10회 최적화(시드 0-4, 두 공간) 43.1 s: 쌍곡 재시작 평균 8.6 s (Adam 10870회 + 마무리 536회), 유클리드 평균 0.1 s (Adam 407회)
합성 점 n=250(쌍 31,125): 쌍곡 Adam 한 단계 7.73 ms, 함수·기울기 7.52 ms; 유클리드 1.53 / 1.46 ms
임베딩 입력 14개 x (쌍곡 5 + 유클리드 5) 재시작


,쌍곡 재시작(초),유클리드 재시작(초),직렬 합(시간),"병렬 10개(시간, 이상적)","재실행 최악: 쌍곡 11회(병렬, 시간)"
추정,,,,,
A 쌍 수 비례(tree 시간 x 65),559,5,10.96,1.10,2.40
B n=250 한 반복 시간 x tree 반복 수,88,1,1.72,0.17,0.38
C 반복 상한(Adam 5만 + 마무리 2만),537,106,12.50,1.25,2.50


## 3절. 실데이터 δ·δ̄·임베딩·판정 (`RUN_REAL = True`일 때만. 1부에서는 코드만 있고 건너뛴다)

각 셀은 `embed_lib` 함수만 부른다. 결과는 `C:\dev\kfm-data\spike\s4\s4_real\`에 둔다. 재시작 결과는 `s4\restart_cache\`에 쌓이므로(입력·초기값·설정이 같으면 재사용) 중간에 멈춰도 이어서 돌릴 수 있다.

- 3-1: `d_time` 250·249·247개 단위와 `d_flow` 주·대안의 δ, δ_rel, δ̄ (전체, 섬 연결 지역 제외).
- 3-2: 대조군 20개의 δ̄ (주·대안 정규화, 전체·섬 제외).
- 3-3: 임베딩 14개 입력(`d_time` 250·247, `d_flow` 주·대안, 대조군 앞 5개의 주·대안), 각각 쌍곡·유클리드 5회 재시작(실패 시 재실행).
- 3-4: S4 보고 항목: 평균 상대 왜곡, 비(쌍곡/유클리드), stress 비, 수렴 여부, 반복 수, 상한에 닿은 점 수, 최대 반지름, 최악 20쌍.
- 3-5: 판정(ADR 0010 7항): 중력모형 이상의 위계(4개 조건), 쌍곡 이점(`d_time`, `d_flow` 주·대안).
- 3-6: 실행 기록(config_hash, 자료 MD5, 코드 버전, 시드).

In [21]:
# 3-1. δ 진단: d_time 250·249·247, d_flow 주·대안 (전체와 섬 연결 지역 제외 부분집합)
N_CTRL_DELTA, N_CTRL_EMBED = 20, 5   # δ̄에 쓰는 대조군 수, 임베딩에 쓰는 대조군 수(ADR 0010 6항)
OUT_REAL = OUT / "s4_real"
if RUN_REAL:
    # 실행 전 고정 커밋 확인(ADR 0013 7항): embed_lib.py와 s4_freeze.json이 커밋되어 있고 바뀌지 않았어야 한다
    st_ = subprocess.run(["git", "status", "--porcelain", "--", "embed_lib.py", "s4_freeze.json"], capture_output=True, text=True, cwd=NB_DIR)
    assert st_.returncode == 0 and st_.stdout.strip() == "", f"embed_lib.py와 s4_freeze.json을 먼저 커밋해야 한다: {st_.stdout!r} {st_.stderr!r}"
    OUT_REAL.mkdir(exist_ok=True)
    DELTA = el.delta_table(DATA_REAL["delta_inputs"], CFG)
    DELTA.to_csv(OUT_REAL / "delta_table.csv", index=False, encoding="utf-8-sig")
    display(DELTA[["입력", "부분집합", "n", "diam", "delta", "delta_rel", "delta_bar", "n_ties", "delta_diff", "seconds"]])
else:
    print("RUN_REAL = False: 3-1 건너뜀")

,입력,부분집합,n,diam,delta,delta_rel,delta_bar,n_ties,delta_diff,seconds
0,d_time_250,전체,250,621.429746,146.973810,0.473018,0.242575,0,0.000000e+00,0.312507
1,d_time_250,섬 제외,247,428.615185,100.310427,0.468068,0.241917,0,2.842171e-14,0.301100
2,d_time_249,전체,249,621.429746,146.973810,0.473018,0.242951,0,0.000000e+00,0.362694
3,d_time_249,섬 제외,246,428.615185,100.310427,0.468068,0.242291,0,2.842171e-14,0.308565
4,d_time_247,전체,247,508.284133,100.310427,0.394702,0.242778,0,2.842171e-14,0.326915
5,d_time_247,섬 제외,246,428.615185,100.310427,0.468068,0.242291,0,2.842171e-14,0.323160
6,d_flow_main,전체,247,26.531934,9.581529,0.722264,0.301166,18628,1.776357e-15,0.357179
7,d_flow_main,섬 제외,246,26.531934,9.581529,0.722264,0.300715,18132,1.776357e-15,0.360210
8,d_flow_alt,전체,247,26.269502,9.407934,0.716263,0.300724,21032,3.552714e-15,0.351581
9,d_flow_alt,섬 제외,246,26.269502,9.407934,0.716263,0.300306,20548,3.552714e-15,0.363887


In [22]:
# 3-2. 대조군 δ̄ (두 정규화, 전체와 섬 제외)
if RUN_REAL:
    CTRL_DB = el.control_delta_bars({nm: DATA_REAL["controls"][nm][:N_CTRL_DELTA] for nm in el.NORMS}, DATA_REAL["flow_island"], CFG)
    CTRL_SUMMARY = pd.DataFrame([{"정규화": k[0], "부분집합": k[1], "n": len(v), "최솟값": v.min(), "중앙값": np.median(v), "최댓값": v.max()} for k, v in CTRL_DB.items()])
    np.savez(OUT_REAL / "control_delta_bars.npz", **{f"{a}|{b}": v for (a, b), v in CTRL_DB.items()})
    display(CTRL_SUMMARY)
else:
    print("RUN_REAL = False: 3-2 건너뜀")

,정규화,부분집합,n,최솟값,중앙값,최댓값
0,main,전체,20,0.259798,0.262145,0.264449
1,main,섬 제외,20,0.259237,0.261542,0.263825
2,alt,전체,20,0.262349,0.265324,0.266794
3,alt,섬 제외,20,0.261703,0.264607,0.266125


In [23]:
# 3-3. 임베딩: 14개 입력 x (쌍곡 5 + 유클리드 5) 재시작. 프로세스 풀(각 1스레드)에서 돌린다.
if RUN_REAL:
    PROBLEMS, CTRL_NAMES = el.embedding_problems(DATA_REAL, n_ctrl_embed=N_CTRL_EMBED, embed_alt=True)
    t0 = time.perf_counter()
    RUNNER = el.PoolRunner(N_WORKERS, cache_dir=OUT / "restart_cache")
    try:
        EMB = el.embed_many(PROBLEMS, RUNNER, CFG)
    finally:
        RUNNER.close()
    EMB_SECONDS = time.perf_counter() - t0
    el.save_embeddings(EMB, OUT_REAL, labels=LABELS, cfg=CFG, D_by_name=PROBLEMS)
    EMB_TABLE = el.embedding_table(EMB)
    EMB_TABLE.to_csv(OUT_REAL / "embedding_table.csv", index=False, encoding="utf-8-sig")
    print(f"임베딩 {len(EMB)}개 입력, 실행 {EMB_SECONDS / 60:.1f} 분 (재시작 합 {EMB_TABLE['초(재시작 합)'].sum() / 3600:.2f} 시간)")
else:
    print("RUN_REAL = False: 3-3 건너뜀")

임베딩 14개 입력, 실행 9.7 분 (재시작 합 1.48 시간)


In [24]:
# 3-4. S4 보고 항목: 평균 상대 왜곡, 비, stress 비, 수렴 여부, 반복 수, 상한에 닿은 점 수, 최대 반지름, 최악 20쌍
if RUN_REAL:
    display(EMB_TABLE)
    for nm_ in ("d_time_250", "d_time_247", "d_flow_main", "d_flow_alt"):
        display(el.restart_table(EMB[nm_]).assign(입력=nm_))
    for nm_ in ("d_time_250", "d_flow_main"):
        for tag_ in ("hyp", "euc"):
            path_ = OUT_REAL / f"worst20_{nm_}_{tag_}.csv"
            if path_.exists():
                print(f"[{nm_}, {'쌍곡' if tag_ == 'hyp' else '유클리드'}] 최악 20쌍")
                display(pd.read_csv(path_))
else:
    print("RUN_REAL = False: 3-4 건너뜀")

,입력,n,기록,쌍곡 이점,평균 상대 왜곡 쌍곡,평균 상대 왜곡 유클리드,왜곡 비,stress 비(판정 재시작),stress 비(전체 최솟값),수렴 쌍곡/유클리드,마무리 수렴(판정 재시작) 쌍곡,마무리 수렴(판정 재시작) 유클리드,반복 수 쌍곡(Adam+마무리),반복 수 유클리드(Adam+마무리),상한에 닿은 점 수,최대 반지름,시도 수,초(재시작 합)
0,d_time_250,250,쌍곡 이점 없음,쌍곡 이점 없음,0.071606,0.073127,0.979199,0.977587,0.977587,"5/5, 5/5",True,True,"(453, 391)","(510, 25)",0,2.194218,1,162.335856
1,d_time_247,247,쌍곡 이점 없음,쌍곡 이점 없음,0.071714,0.073259,0.978907,0.974178,0.974178,"5/5, 5/5",True,True,"(1632, 412)","(736, 28)",0,2.259775,1,151.859278
2,d_flow_main,247,쌍곡 이점 없음,쌍곡 이점 없음,0.146072,0.160035,0.912752,0.874717,0.874717,"5/5, 5/5",True,True,"(4949, 870)","(666, 1)",1,5.000000,1,335.727952
3,d_flow_alt,247,쌍곡 이점 없음,쌍곡 이점 없음,0.148828,0.158112,0.941285,0.922816,0.922816,"5/5, 5/5",True,True,"(2837, 233)","(531, 62)",0,4.516214,1,344.521615
4,ctrl_main_0,247,쌍곡 이점 없음(평탄 극한),쌍곡 이점 없음(평탄 극한),0.089031,0.089010,1.000243,1.000255,1.000255,"3/5, 5/5",True,True,"(1510, 5611)","(385, 1)",0,0.001592,1,203.800396
5,ctrl_main_1,247,쌍곡 이점 없음(평탄 극한),쌍곡 이점 없음(평탄 극한),0.089867,0.089864,1.000031,1.000068,1.000068,"5/5, 5/5",True,True,"(2291, 1851)","(419, 1)",0,0.000475,1,234.185867
6,ctrl_main_2,247,쌍곡 이점 없음(평탄 극한),쌍곡 이점 없음(평탄 극한),0.090805,0.090805,1.000001,1.000001,1.000001,"3/5, 5/5",True,True,"(1382, 15149)","(367, 1)",0,0.006264,1,373.416178
7,ctrl_main_3,247,쌍곡 이점 없음,쌍곡 이점 없음,0.092969,0.093022,0.999427,0.999997,0.999997,"5/5, 5/5",True,True,"(1686, 9540)","(652, 1)",0,0.002033,1,394.154580
8,ctrl_main_4,247,쌍곡 이점 없음(평탄 극한),쌍곡 이점 없음(평탄 극한),0.093123,0.093122,1.000012,1.000002,1.000002,"5/5, 5/5",True,True,"(2045, 11236)","(415, 1)",0,0.006532,1,446.604552
9,ctrl_alt_0,247,쌍곡 이점 없음(평탄 극한),쌍곡 이점 없음(평탄 극한),0.093277,0.093266,1.000112,1.000047,1.000047,"5/5, 5/5",True,True,"(1499, 6570)","(498, 1)",0,0.005183,1,469.704351


,attempt,space,init,seed,m,stress,distortion,finish_converged,adam_converged,n_iter_adam,n_iter_finish,n_at_cap,max_radius,seconds,입력
0,0,euclid,mds,NaN,NaN,319.465208,0.073127,True,True,510,25,NaN,NaN,3.434826,d_time_250
1,0,euclid,U,0.0,NaN,319.848951,0.073131,True,True,945,31,NaN,NaN,2.895393,d_time_250
2,0,euclid,U,1.0,NaN,319.571787,0.073152,True,True,726,26,NaN,NaN,2.201671,d_time_250
3,0,euclid,U,2.0,NaN,319.848951,0.073131,True,True,1405,15,NaN,NaN,4.013054,d_time_250
4,0,euclid,U,3.0,NaN,321.493649,0.073297,True,True,882,23,NaN,NaN,2.793367,d_time_250
5,0,poincare,U,0.0,NaN,450.265671,0.084704,True,True,2218,567,0.0,1.999930,37.639778,d_time_250
6,0,poincare,U,1.0,NaN,312.304931,0.071606,True,True,1600,341,0.0,2.208549,26.430477,d_time_250
7,0,poincare,U,2.0,NaN,312.798148,0.071614,True,True,2394,495,0.0,2.163440,38.805990,d_time_250
8,0,poincare,shrunk_m1,NaN,0.5,312.359279,0.071609,True,True,2252,258,0.0,2.208499,32.821752,d_time_250
9,0,poincare,shrunk_m2,NaN,0.9,312.304930,0.071606,True,True,453,391,0.0,2.194218,11.299547,d_time_250


,attempt,space,init,seed,m,stress,distortion,finish_converged,adam_converged,n_iter_adam,n_iter_finish,n_at_cap,max_radius,seconds,입력
0,0,euclid,mds,NaN,NaN,313.485282,0.073384,True,True,649,31,NaN,NaN,2.149189,d_time_247
1,0,euclid,U,0.0,NaN,312.827264,0.073300,True,True,944,27,NaN,NaN,2.949348,d_time_247
2,0,euclid,U,1.0,NaN,311.984884,0.073259,True,True,736,28,NaN,NaN,2.142306,d_time_247
3,0,euclid,U,2.0,NaN,447.314862,0.086143,True,True,1389,17,NaN,NaN,4.090169,d_time_247
4,0,euclid,U,3.0,NaN,312.854957,0.073320,True,True,816,23,NaN,NaN,2.429328,d_time_247
5,0,poincare,U,0.0,NaN,441.235717,0.084929,True,True,1716,487,0.0,2.003608,30.089739,d_time_247
6,0,poincare,U,1.0,NaN,303.928920,0.071714,True,True,1632,412,0.0,2.259775,27.584227,d_time_247
7,0,poincare,U,2.0,NaN,303.928920,0.071714,True,True,2066,354,0.0,2.173305,32.745871,d_time_247
8,0,poincare,shrunk_m1,NaN,0.5,303.988941,0.071716,True,True,2506,334,0.0,2.279442,36.592309,d_time_247
9,0,poincare,shrunk_m2,NaN,0.9,304.092820,0.071721,True,True,611,230,0.0,2.256052,11.086790,d_time_247


,attempt,space,init,seed,m,stress,distortion,finish_converged,adam_converged,n_iter_adam,n_iter_finish,n_at_cap,max_radius,seconds,입력
0,0,euclid,mds,NaN,NaN,1364.081480,0.159911,True,True,415,1,NaN,NaN,1.138640,d_flow_main
1,0,euclid,U,0.0,NaN,1363.916061,0.160154,True,True,470,1,NaN,NaN,1.332059,d_flow_main
2,0,euclid,U,1.0,NaN,1364.064159,0.160170,True,True,460,1,NaN,NaN,1.324979,d_flow_main
3,0,euclid,U,2.0,NaN,1363.976715,0.160277,True,True,455,1,NaN,NaN,1.369299,d_flow_main
4,0,euclid,U,3.0,NaN,1363.439937,0.160035,True,True,666,1,NaN,NaN,1.920129,d_flow_main
5,0,poincare,U,0.0,NaN,1192.623678,0.146072,True,True,4949,870,1.0,5.000000,76.816666,d_flow_main
6,0,poincare,U,1.0,NaN,1655.233390,0.174544,True,True,7123,1527,28.0,5.000000,114.438565,d_flow_main
7,0,poincare,U,2.0,NaN,1199.575007,0.146407,True,True,4440,1241,1.0,5.000000,76.273205,d_flow_main
8,0,poincare,shrunk_m1,NaN,0.5,1193.249255,0.146243,True,True,2126,223,0.0,4.955481,30.003286,d_flow_main
9,0,poincare,shrunk_m2,NaN,0.9,1200.074224,0.146499,True,True,1607,828,1.0,5.000000,31.111124,d_flow_main


,attempt,space,init,seed,m,stress,distortion,finish_converged,adam_converged,n_iter_adam,n_iter_finish,n_at_cap,max_radius,seconds,입력
0,0,euclid,mds,NaN,NaN,1334.405355,0.158112,True,True,531,62,NaN,NaN,3.706975,d_flow_alt
1,0,euclid,U,0.0,NaN,1334.761957,0.158115,True,True,516,1,NaN,NaN,3.335243,d_flow_alt
2,0,euclid,U,1.0,NaN,1334.750028,0.158178,True,True,422,1,NaN,NaN,3.027814,d_flow_alt
3,0,euclid,U,2.0,NaN,1335.661360,0.158405,True,True,439,1,NaN,NaN,3.079679,d_flow_alt
4,0,euclid,U,3.0,NaN,1335.519480,0.157971,True,True,673,1,NaN,NaN,3.983467,d_flow_alt
5,0,poincare,U,0.0,NaN,1231.561254,0.148851,True,True,6375,284,0.0,4.673926,86.851757,d_flow_alt
6,0,poincare,U,1.0,NaN,1231.663379,0.148864,True,True,6417,403,0.0,4.528321,87.132796,d_flow_alt
7,0,poincare,U,2.0,NaN,1237.794600,0.149008,True,True,5719,720,0.0,4.584524,81.195272,d_flow_alt
8,0,poincare,shrunk_m1,NaN,0.5,1231.410704,0.148828,True,True,2837,233,0.0,4.516214,39.469565,d_flow_alt
9,0,poincare,shrunk_m2,NaN,0.9,1237.955877,0.148916,True,True,2312,234,0.0,4.484892,32.739049,d_flow_alt


...

,i,j,name_i,name_j,d_true,d_emb,rel_err,abs_rel_err
0,220,228,강원특별자치도 동해시,강원특별자치도 정선군,88.665185,14.169737,-0.840188,0.840188
1,167,215,전라남도 완도군,제주특별자치도 제주시,201.674797,352.646837,0.748592,0.748592
2,205,208,경상남도 의령군,경상남도 고성군,59.905820,15.396110,-0.742995,0.742995
3,223,228,강원특별자치도 삼척시,강원특별자치도 정선군,89.165541,27.922835,-0.686843,0.686843
4,230,231,강원특별자치도 화천군,강원특별자치도 양구군,61.506929,20.689688,-0.663620,0.663620
5,199,200,경상남도 통영시,경상남도 사천시,54.717506,18.538787,-0.661191,0.661191
6,200,208,경상남도 사천시,경상남도 고성군,49.184653,18.547036,-0.622910,0.622910
7,200,210,경상남도 사천시,경상남도 하동군,51.636614,20.267190,-0.607504,0.607504
8,199,205,경상남도 통영시,경상남도 의령군,63.212835,25.093971,-0.603024,0.603024
9,49,181,대구광역시 군위군,경상북도 의성군,38.449419,15.381725,-0.599949,0.599949


[d_flow_main, 쌍곡] 최악 20쌍


,i,j,name_i,name_j,d_true,d_emb,rel_err,abs_rel_err
0,122,123,충청북도 충주시,충청북도 제천시,2.055568,5.251479,1.554758,1.554758
1,186,213,경상북도 고령군,경상남도 합천군,3.075953,7.829542,1.545404,1.545404
2,129,179,충청북도 괴산군,경상북도 문경시,4.575263,10.851093,1.371687,1.371687
3,126,173,충청북도 영동군,경상북도 김천시,2.906074,6.834281,1.351723,1.351723
4,135,143,충청남도 보령시,충청남도 서천군,2.473320,5.365861,1.169497,1.169497
5,124,178,충청북도 보은군,경상북도 상주시,4.933809,10.688049,1.166288,1.166288
6,184,191,경상북도 영덕군,경상북도 울진군,2.121233,4.536228,1.138487,1.138487
7,191,220,경상북도 울진군,강원특별자치도 삼척시,2.205739,4.690340,1.126426,1.126426
8,170,172,경상북도 포항시 남구,경상북도 경주시,1.829309,3.848522,1.103812,1.103812
9,93,148,경기도 고양시 일산동구,전라남도 목포시,8.626863,17.891315,1.073907,1.073907


[d_flow_main, 유클리드] 최악 20쌍


,i,j,name_i,name_j,d_true,d_emb,rel_err,abs_rel_err
0,202,204,경상남도 거제시,경상남도 의령군,6.323679,0.661882,-0.895333,0.895333
1,165,168,전라남도 영광군,전라남도 진도군,7.317229,0.781762,-0.893162,0.893162
2,153,160,전라남도 담양군,전라남도 강진군,6.165275,0.670238,-0.891288,0.891288
3,14,93,서울특별시 양천구,경기도 고양시 일산동구,4.545227,0.522439,-0.885058,0.885058
4,190,191,경상북도 봉화군,경상북도 울진군,4.975453,0.582468,-0.882932,0.882932
5,11,95,서울특별시 은평구,경기도 과천시,6.099199,0.714114,-0.882917,0.882917
6,6,112,서울특별시 중랑구,경기도 양주시,4.828583,0.582996,-0.879261,0.879261
7,28,203,부산광역시 영도구,경상남도 양산시,4.485025,0.557822,-0.875626,0.875626
8,169,245,전라남도 신안군,전북특별자치도 고창군,7.794385,1.020883,-0.869023,0.869023
9,78,108,경기도 수원시 팔달구,경기도 안성시,5.326213,0.701774,-0.868241,0.868241


In [25]:
# 3-5. 판정 (ADR 0010 7항). 결과로 기준값을 바꾸지 않는다.
if RUN_REAL:
    JUDGE = el.judge_all(DELTA, CTRL_DB, EMB, DATA_REAL["flow_names"], CTRL_NAMES, CFG, n_required=N_CTRL_DELTA)
    JUDGE["hierarchy"]["table"].to_csv(OUT_REAL / "hierarchy_table.csv", index=False, encoding="utf-8-sig")
    display(JUDGE["hierarchy"]["table"])
    print("중력모형 이상의 위계(4개 조건 모두 통과):", JUDGE["hierarchy"]["hierarchy"])
    for norm_, j_ in JUDGE["flow_advantage"].items():
        print(f"d_flow({'주' if norm_ == 'main' else '대안'}) 쌍곡 이점: {j_['advantage']} | {j_['reason']}")
    for nm_ in ("d_time_250", "d_time_247"):
        s_ = EMB[nm_]["summary"]
        print(f"{nm_} 쌍곡 이점: {s_['advantage']} | 기록 {s_['record']} | {s_['reason']}")
    emit_like = {"hierarchy": JUDGE["hierarchy"]["hierarchy"], "flow_advantage": JUDGE["flow_advantage"],
                 "d_time_advantage": {nm_: EMB[nm_]["summary"]["advantage"] for nm_ in ("d_time_250", "d_time_247")}}
    (OUT_REAL / "judge.json").write_text(json.dumps(el.to_jsonable(emit_like), ensure_ascii=False, indent=1), encoding="utf-8")
else:
    print("RUN_REAL = False: 3-5 건너뜀")

,정규화,부분집합,d_flow,대조군 최솟값,대조군 중앙값,대조군 최댓값,d_flow 이하인 대조군 수,여유(최솟값 - d_flow),통과
0,main,전체,0.301166,0.259798,0.262145,0.264449,20,-0.041368,False
1,main,섬 제외,0.300715,0.259237,0.261542,0.263825,20,-0.041478,False
2,alt,전체,0.300724,0.262349,0.265324,0.266794,20,-0.038376,False
3,alt,섬 제외,0.300306,0.261703,0.264607,0.266125,20,-0.038603,False


중력모형 이상의 위계(4개 조건 모두 통과): False
d_flow(주) 쌍곡 이점: 쌍곡 이점 없음 | 평균 상대 왜곡 비 0.9128 (문턱 0.8)
d_flow(대안) 쌍곡 이점: 쌍곡 이점 없음 | 평균 상대 왜곡 비 0.9413 (문턱 0.8)
d_time_250 쌍곡 이점: 쌍곡 이점 없음 | 기록 쌍곡 이점 없음 | 평균 상대 왜곡 비 0.9792 (문턱 0.8)
d_time_247 쌍곡 이점: 쌍곡 이점 없음 | 기록 쌍곡 이점 없음 | 평균 상대 왜곡 비 0.9789 (문턱 0.8)


In [26]:
# 3-6. 실행 기록 (config_hash, data_snapshot, code_version, seed)
if RUN_REAL:
    git_head = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, cwd=NB_DIR).stdout.strip()
    RUN_RECORD = {"config_hash": el.config_hash(CFG), "config": CFG, "data_snapshot": {"inputs_md5": MD5, "outputs_md5": OUT_MD5},
                  "code_version": {"git_head": git_head, "embed_lib.py": md5_file(NB_DIR / "embed_lib.py")},
                  "seed": SEED_RULES, "n_workers": N_WORKERS, "embedding_seconds": EMB_SECONDS,
                  "attempts_used": {nm_: EMB[nm_]["final_attempt"] + 1 for nm_ in EMB}}
    (OUT_REAL / "runs_record.json").write_text(json.dumps(el.to_jsonable(RUN_RECORD), ensure_ascii=False, indent=1), encoding="utf-8")
    print("실행 기록 저장:", OUT_REAL / "runs_record.json")
else:
    print("RUN_REAL = False: 3-6 건너뜀")
print(f"노트북 전체 실행 {time.time() - T_START:.0f} s")

실행 기록 저장: C:\dev\kfm-data\spike\s4\s4_real\runs_record.json
노트북 전체 실행 729 s
